In [2]:
from epyt import epanet
import pandas as pd
import numpy as np
import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Epanet

In [41]:
import os
arquivo = 'Epanet Gerados\\Recanto_Riacho2\\Recanto_Riacho Ajuste V2 - 09.09.inp'
print(os.path.exists(arquivo))

True


In [17]:
import chardet

def analisar_inp(caminho_arquivo):
    problemas = []

    # Detectar encoding do arquivo
    with open(caminho_arquivo, "rb") as f:
        raw = f.read()
        resultado = chardet.detect(raw)
        encoding_detectado = resultado['encoding']
        confianca = resultado['confidence']

    print(f"📌 Encoding detectado: {encoding_detectado} (confiança {confianca:.2f})")

    # Reabre com encoding detectado (ou força utf-8 se der erro)
    try:
        with open(caminho_arquivo, "r", encoding=encoding_detectado) as f:
            linhas = f.readlines()
    except:
        with open(caminho_arquivo, "r", encoding="utf-8", errors="replace") as f:
            linhas = f.readlines()

    # Percorre linha por linha
    for num, linha in enumerate(linhas, start=1):
        # Procura caracteres não ASCII
        for char in linha:
            if ord(char) > 127:  # fora do ASCII básico
                problemas.append((num, char, linha.strip()))
                break  # reporta só uma vez por linha

    if problemas:
        print("\n⚠️ Problemas encontrados (linhas com acentos/caracteres especiais):")
        for num, char, conteudo in problemas:
            print(f"  Linha {num}: caractere '{char}' → {conteudo}")
    else:
        print("\n✅ Nenhum caractere problemático encontrado.")

# Exemplo de uso
analisar_inp("Epanet Cavitacao\\Samambaia\\NOVAS VRPs 0106.inp")
# analisar_inp("Epanet Cavitacao\\Gama 2\\GAM2 Novas VRP.inp")
# analisar_inp("Epanet Gerados\\Recanto_Riacho2\\Reborn\\RCE RF2 Reborn.inp")



📌 Encoding detectado: ascii (confiança 1.00)

✅ Nenhum caractere problemático encontrado.


In [36]:
d = epanet("Epanet Cavitacao\\Samambaia\\NOVAS VRPs 01.06.inp")
# d = epanet('Epanet Cavitacao\\Gama 2\\GAM2 Novas VRP.inp')
# d = epanet('Epanet Cavitacao\\Gama 2\\GAM2 Novas VRP+VRP.GAM.013.inp')
# d = epanet("Epanet Gerados\\Recanto_Riacho2\\Reborn\\RCE RF2 Reborn.inp")

# d = epanet('Epanet\\Vicente\\REV 01\\sugestao_de_novas_VRPs_FINAL.inp')
# d = epanet('Epanet\\Vicente\\REV 01\\Cenario_01_VCP_sem_AB.inp')



EPANET version 20200 loaded (EPyT version v1.2.1 - Last Update: 09/01/2024).
Input File NOVAS VRPs 01.06.inp loaded successfully.



# Plotagem

In [450]:
d.getNodeBaseDemands(dict_name_index['2316'])

{1: array([0.18887299])}

In [26]:
# Plot links IDs
# d.plot(linksID=True)


In [318]:
#Criando uma biblioteca para puxar os IDs corretos

ids_nós = {nome:index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
ids_nós

{'1215': 1,
 '1630': 2,
 '0': 3,
 '425': 4,
 '1473': 5,
 '6': 6,
 '1763': 7,
 '673': 8,
 '2499': 9,
 '2261': 10,
 '2505': 11,
 '1197': 12,
 '198': 13,
 '1499': 14,
 '1625': 15,
 '2354': 16,
 '3183': 17,
 '235': 18,
 '1602': 19,
 '2298': 20,
 '286': 21,
 '833': 22,
 '231': 23,
 '1284': 24,
 '802': 25,
 '1214': 26,
 '281': 27,
 '1263': 28,
 '2083': 29,
 '1494': 30,
 '306': 31,
 '553': 32,
 '2289': 33,
 '2108': 34,
 '1445': 35,
 '282': 36,
 '7': 37,
 '1194': 38,
 '196': 39,
 '1997': 40,
 '2233': 41,
 '426': 42,
 '656': 43,
 '3090': 44,
 '2361': 45,
 '2136': 46,
 '387': 47,
 '1830': 48,
 '1835': 49,
 '791': 50,
 '446': 51,
 '2364': 52,
 '501': 53,
 '2238': 54,
 '1626': 55,
 '2121': 56,
 '507': 57,
 '1137': 58,
 '1597': 59,
 '789': 60,
 '2173': 61,
 '1196': 62,
 '1579': 63,
 '2228': 64,
 '2032': 65,
 '1983': 66,
 '1771': 67,
 '1520': 68,
 '2244': 69,
 '1332': 70,
 '650': 71,
 '1535': 72,
 '39': 73,
 '1505': 74,
 '2522': 75,
 '870': 76,
 '836': 77,
 '2358': 78,
 '508': 79,
 '649': 80,
 '343'

In [319]:
d.getNodeElevations(ids_nós["553"])

1129.219970703125

In [320]:
nome_nó = d.getNodeNameID(ids_nós['553'])
nome_nó

'553'

In [321]:
d.runsCompleteSimulation()

d.getNodePressure(ids_nós['553'])

33.78041458129883

In [ ]:
d.getvalve

# Analisando VRPs

In [50]:
d.getNodesConnectingLinksID(d.getLinkValveNameID([1,2]))

array([['PM.VRP.VCP.015', '9288'],
       ['PM.VRP.VCP.026', '11286']], dtype='<U14')

In [ ]:
#Abrindo uma analise Hidraúlica para printar a pressão horária de um nó

d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()
tstep,P , T_H, D, H, F, S, = 1, [], [], [], [] ,[], []
tempo = []
dados = []
while (tstep>0):
    t = d.runHydraulicAnalysis()
    P.append(d.getNodePressure())
    D.append(d.getNodeActualDemand())
    H.append(d.getNodeHydraulicHead())
    S.append(d.getLinkStatus())
    F.append(d.getLinkFlows())
    T_H.append(t)
    tstep=d.nextHydraulicAnalysisStep()
    tempo.append(tstep)

    # dados.append([tstep, P, D, H, F, S])
d.closeHydraulicAnalysis()

dict_index_name = {index:nome for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
dict_name_index = {nome:index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}



for tempo, pressao in enumerate(P):
    tempo_formatado = str(datetime.timedelta(hours=tempo))
    print(tempo_formatado, pressao[dict_name_index["553"]-1])
    # print(dict_name_index)


0:00:00 36.88386535644531
1:00:00 36.906944274902344
2:00:00 37.436737060546875
3:00:00 37.129573822021484
4:00:00 37.45825958251953
5:00:00 37.29825973510742
6:00:00 36.95606231689453
7:00:00 36.71164321899414
8:00:00 36.8940544128418
9:00:00 36.959754943847656
10:00:00 36.57255172729492
11:00:00 36.323455810546875
12:00:00 37.0860595703125
13:00:00 36.44062423706055
14:00:00 36.49254608154297
15:00:00 37.37702178955078
16:00:00 36.753047943115234
17:00:00 37.280975341796875
18:00:00 36.89097595214844
19:00:00 36.39506530761719
20:00:00 36.970977783203125
21:00:00 37.02733612060547
22:00:00 36.94160461425781
23:00:00 36.550697326660156
1 day, 0:00:00 36.61386489868164


In [75]:
def obter_nós(nome_valvula):
    dict_valve_name_index = {nome:index for index, nome in enumerate(d.getLinkValveNameID(), start=1)}
    valor = d.getNodesConnectingLinksID(d.getLinkValveNameID([dict_valve_name_index.get(nome_valvula)]))
    return valor

In [675]:
d.getLinkValveNameID()

['VRP.VCP.015',
 'VRP.VCP.025',
 'VRP.VCP.017',
 'VRP.VCP.018',
 'VRP.VCP.019',
 'VRP.VCP.021',
 'VRP.VCP.020',
 'VRP.VCP.006',
 'VRP.VCP.032',
 'VRP.VCP.033',
 'VRP.VCP.009',
 'VRP.VCP.010',
 'VRP.VCP.001',
 'VRP.VCP.011',
 'VRP.VCP.012',
 'VRP.VCP.039',
 'VRP.VCP.034',
 'VRP.VCP.013',
 'VRP.VCP.038',
 'VRP.VCP.036',
 'VRP.VCP.027',
 'VRP.VCP.022',
 'VRP.VCP.023',
 'VRP.VCP.035',
 'VRP.VCP.037',
 'VRP.VCP.024',
 'VRP.VCP.007',
 'VRP.VCP.008',
 'VRP.VCP.002',
 'VRP.VCP.028',
 'VRP.VCP.005',
 'VRP.VCP.014',
 'VRP.VCP.031',
 'VRP.VCP.003',
 'VRP.VCP.029',
 'VRP.VCP.004',
 'VRP.VCP.030',
 'ControlePressaoBomba',
 'VRP.VCP.040',
 'VRP.VCP.041',
 'VRP.VCP.AcaODIRETA01',
 'VRP.VCP.AcaODIRETA02',
 'VRP.VCP.042',
 'VRP.VCP.026']

In [68]:
# obter_nós(nome_valvula = "VRP.VCP.NOVA02")


In [70]:
d.getNodesConnectingLinksID(dict_name_index['11160'])


array([['5589', '5590']], dtype='<U4')

# Criando tabela de média horaria

In [ ]:
import datetime
import pandas as pd

# ==========================================================
# Rodar análise hidráulica
# ==========================================================
d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()

tstep = 1

media_horaria = []
horarios = []

while tstep > 0:

    t = d.runHydraulicAnalysis()

    # ------------------------------------------------------
    # Salvar somente horários inteiros
    # t vem em segundos
    # 3600 = 1 hora
    # ------------------------------------------------------
    if t % 3600 == 0:

        pressao = d.getNodePressure()

        media = round(
            sum(pressao) / len(pressao),
            2
        )

        media_horaria.append(media)

        horarios.append(
            str(datetime.timedelta(seconds=t))
        )

    tstep = d.nextHydraulicAnalysisStep()

d.closeHydraulicAnalysis()

# ==========================================================
# Média geral
# ==========================================================
media_geral = round(
    sum(media_horaria) / len(media_horaria),
    2
)

# ==========================================================
# Criar DataFrame
# ==========================================================
df = pd.DataFrame({
    "Hora": horarios,
    "Média Horária (m.c.a.)": media_horaria
})

# linha final
df.loc[len(df)] = [
    "Média Geral",
    media_geral
]

# visualizar
df.head(26)

,Hora,Média Horária (m.c.a.)
0,0:00:00,25.24
1,1:00:00,25.52
2,2:00:00,25.94
3,3:00:00,26.01
4,4:00:00,26.15
5,5:00:00,26.15
6,6:00:00,26.19
7,7:00:00,25.91
8,8:00:00,25.55
9,9:00:00,25.10


In [38]:
df.to_excel('Media horaria\\Samambaia\\SAM Sugestão final.xlsx')

In [41]:
import datetime
import numpy as np
import pandas as pd

# ==========================================================
# CONFIGURAÇÃO DAY-NIGHT
# ==========================================================
# Preencher apenas as VRPs que possuem atuação Day-Night

vrps_day_night = {
    'VRP.SAM.009': {
        'inicio': 22,
        'fim': 6
    },

    'VRP.SAM.012': {
        'inicio': 22,
        'fim': 6
    },
        'VRP.SAM.026': {
        'inicio': 22,
        'fim': 6
    }
}

# ==========================================================
# FUNÇÃO PARA OBTER NÓS DA VRP
# ==========================================================
def obter_nos_da_valvula(nome_valvula):

    dict_valve_name_index = {
        nome: index
        for index, nome in enumerate(
            d.getLinkValveNameID(),
            start=1
        )
    }

    valve_id = d.getLinkValveNameID(
        [dict_valve_name_index[nome_valvula]]
    )

    nos_conectados = d.getNodesConnectingLinksID(
        valve_id
    )

    if (
        isinstance(nos_conectados, np.ndarray)
        and nos_conectados.ndim > 1
    ):
        nos_conectados = nos_conectados[0]

    return [str(no) for no in nos_conectados]

# ==========================================================
# DICIONÁRIO NOME -> ÍNDICE
# ==========================================================
dict_name_index = {
    nome: index
    for nome, index in zip(
        d.getNodeNameID(),
        d.getNodeIndex()
    )
}

# ==========================================================
# LISTA DE VRPs
# ==========================================================
lista_valvulas = [

'VRP.SAM.001',
'VRP.SAM.002',
'VRP.SAM.003',
'VRP.SAM.004',
'VRP.SAM.005',
'VRP.SAM.006',
'VRP.SAM.007',
'VRP.SAM.008',
'VRP.SAM.009',
'VRP.SAM.011',
'VRP.SAM.012',
'VRP.SAM.013',
'VRP.SAM.014',
'VRP.SAM.015',
'VRP.SAM.016',
'VRP.SAM.017',
'VRP.SAM.018R',
'VRP.SAM.019',
'VRP.SAM.020',
'VRP.SAM.021',
'VRP.SAM.022',
'VRP.SAM.023',
'VRP.SAM.024',
'VRP.SAM.025',
'VRP.SAM.026',
'VRP.SAM.027',
'VRP.SAM.028',
'VRP.SAM.029',
'VRP.SAM.030',
'VRP.SAM.NOVA.001',
'VRP.SAM.NOVA.002',
'VRP.SAM.NOVA.003',
'VRP.SAM.NOVA.004',
'VRP.SAM.NOVA.005',
'VRP.SAM.NOVA.006',
'VRP.SAM.NOVA.007',
'VRP.SAM.NOVA.008',
'VRP.SAM.NOVA.009',
'VRP.SAM.NOVA.010',
'VRP.SAM.NOVA.011',
'VRP.SAM.NOVA.012',
'VRP.SAM.NOVA.013',
'VRP.SAM.NOVA.014',
'VRP.SAM.NOVA.015',
'VRP.SAM.NOVA.016',
'VRP.SAM.NOVA.017',
'VRP.SAM.NOVA.018',
'VRP.SAM.NOVA.019',
'VRP.SAM.NOVA.020',
'VRP.SAM.NOVA.021',
'VRP.SAM.NOVA.022',
'VRP.SMT.NOVA.001'

]

# ==========================================================
# RODAR APENAS UMA SIMULAÇÃO
# ==========================================================
print('Rodando simulação...')

d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()

tstep = 1

P = []
HORAS = []

while tstep > 0:

    t = d.runHydraulicAnalysis()

    # salva somente horas cheias
    if t % 3600 == 0:

        P.append(
            d.getNodePressure()
        )

        HORAS.append(
            int(t / 3600)
        )

    tstep = d.nextHydraulicAnalysisStep()

d.closeHydraulicAnalysis()

print(f'{len(HORAS)} horários salvos.')

# ==========================================================
# PROCESSAR VRPs
# ==========================================================
resultado = []

for vrp in lista_valvulas:

    try:

        nos = obter_nos_da_valvula(vrp)

        if len(nos) < 2:
            continue

        # ----------------------------------------------
        # ASSUMINDO:
        # nó 0 = montante
        # nó 1 = jusante
        # ----------------------------------------------
        no_montante = nos[0]
        no_jusante = nos[1]

        idx_montante = (
            dict_name_index[no_montante] - 1
        )

        idx_jusante = (
            dict_name_index[no_jusante] - 1
        )

        pressao_montante = []
        pressao_jusante = []

        pressao_dia = []
        pressao_madrugada = []

        # ----------------------------------------------
        # LOOP DOS HORÁRIOS
        # ----------------------------------------------
        for hora, pressao in zip(HORAS, P):

            p_mont = pressao[idx_montante]
            p_jus = pressao[idx_jusante]

            pressao_montante.append(p_mont)
            pressao_jusante.append(p_jus)

            # ------------------------------------------
            # VRP COM DAY-NIGHT
            # ------------------------------------------
            if vrp in vrps_day_night:
            
                inicio = vrps_day_night[vrp]['inicio']
                fim = vrps_day_night[vrp]['fim']

                hora_dia = hora % 24

                # Exemplo: 22h até 6h
                if inicio > fim:
                
                    eh_madrugada = (
                        hora_dia >= inicio
                        or hora_dia <= fim
                    )

                # Exemplo: 0h até 6h
                else:
                
                    eh_madrugada = (
                        inicio <= hora_dia <= fim
                    )

                if eh_madrugada:
                    pressao_madrugada.append(p_jus)
                else:
                    pressao_dia.append(p_jus)                

        # ----------------------------------------------
        # MÉDIAS
        # ----------------------------------------------
        media_montante = round(
            np.mean(pressao_montante),
            2
        )

        media_jusante = round(
            np.mean(pressao_jusante),
            2
        )

        media_dia = np.nan
        media_madrugada = np.nan

        if vrp in vrps_day_night:

            if len(pressao_dia) > 0:
                media_dia = round(
                    np.mean(pressao_dia),
                    2
                )

            if len(pressao_madrugada) > 0:
                media_madrugada = round(
                    np.mean(pressao_madrugada),
                    2
                )

        resultado.append({

            'VRP': vrp,

            'No_Montante': no_montante,
            'No_Jusante': no_jusante,

            'Pressao_Montante_Media':
                media_montante,

            'Pressao_Jusante_Media':
                media_jusante,

            'Pressao_Jusante_Dia':
                media_dia,

            'Pressao_Jusante_Madrugada':
                media_madrugada

        })

        print(f'OK - {vrp}')

    except Exception as e:

        print(f'Erro em {vrp}: {e}')

# ==========================================================
# DATAFRAME FINAL
# ==========================================================
df_resultado = pd.DataFrame(resultado)
df_resultado


Rodando simulação...
25 horários salvos.
OK - VRP.SAM.001
OK - VRP.SAM.002
OK - VRP.SAM.003
OK - VRP.SAM.004
OK - VRP.SAM.005
OK - VRP.SAM.006
OK - VRP.SAM.007
OK - VRP.SAM.008
OK - VRP.SAM.009
OK - VRP.SAM.011
OK - VRP.SAM.012
OK - VRP.SAM.013
OK - VRP.SAM.014
OK - VRP.SAM.015
OK - VRP.SAM.016
OK - VRP.SAM.017
OK - VRP.SAM.018R
OK - VRP.SAM.019
OK - VRP.SAM.020
OK - VRP.SAM.021
OK - VRP.SAM.022
OK - VRP.SAM.023
OK - VRP.SAM.024
OK - VRP.SAM.025
OK - VRP.SAM.026
OK - VRP.SAM.027
OK - VRP.SAM.028
OK - VRP.SAM.029
OK - VRP.SAM.030
OK - VRP.SAM.NOVA.001
OK - VRP.SAM.NOVA.002
OK - VRP.SAM.NOVA.003
OK - VRP.SAM.NOVA.004
OK - VRP.SAM.NOVA.005
OK - VRP.SAM.NOVA.006
OK - VRP.SAM.NOVA.007
OK - VRP.SAM.NOVA.008
OK - VRP.SAM.NOVA.009
OK - VRP.SAM.NOVA.010
OK - VRP.SAM.NOVA.011
OK - VRP.SAM.NOVA.012
OK - VRP.SAM.NOVA.013
OK - VRP.SAM.NOVA.014
OK - VRP.SAM.NOVA.015
OK - VRP.SAM.NOVA.016
OK - VRP.SAM.NOVA.017
OK - VRP.SAM.NOVA.018
OK - VRP.SAM.NOVA.019
OK - VRP.SAM.NOVA.020
OK - VRP.SAM.NOVA.021
OK 

,VRP,No_Montante,No_Jusante,Pressao_Montante_Media,Pressao_Jusante_Media,Pressao_Jusante_Dia,Pressao_Jusante_Madrugada
0,VRP.SAM.001,PM.VRP.SAM.001,14370,35.74,6.00,NaN,NaN
1,VRP.SAM.002,PM.VRP.SAM.002,13276,81.88,17.00,NaN,NaN
2,VRP.SAM.003,PM.VRP.SAM.003,13959,39.20,7.00,NaN,NaN
3,VRP.SAM.004,PM.VRP.SAM.004,13532,46.86,10.00,NaN,NaN
4,VRP.SAM.005,PM.VRP.SAM.005,9004,74.17,18.00,NaN,NaN
5,VRP.SAM.006,PM.VRP.SAM.006,14424,44.53,7.00,NaN,NaN
6,VRP.SAM.007,PM.VRP.SAM.007,11145,47.41,9.00,NaN,NaN
7,VRP.SAM.008,PM.VRP.SAM.008,8394,42.95,10.00,NaN,NaN
8,VRP.SAM.009,PM.VRP.SAM.009,489,36.09,18.00,22.0,12.0
9,VRP.SAM.011,PM.VRP.SAM.011,13957,38.65,9.00,NaN,NaN


In [42]:
# ==========================================================
# EXPORTAR
# ==========================================================
arquivo_saida = (
    r"Media horaria\Samambaia\Resumo_VRPs SAM.xlsx"
)

df_resultado.to_excel(
    arquivo_saida,
    index=False
)

print('\nArquivo criado com sucesso!')
print(arquivo_saida)


Arquivo criado com sucesso!
Media horaria\Samambaia\Resumo_VRPs SAM.xlsx


# Criando tabela da Cavitação

In [33]:
dict_ids = {index:nome for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}

roughness = d.getLinkRoughnessCoeff()

startnode = [dict_ids[ids[0]] for ids in d.getLinkNodesIndex()]
endnode = [dict_ids[ids[1]] for ids in d.getLinkNodesIndex()]
roughness = d.getLinkRoughnessCoeff()
length = d.getLinkLength()
diameter = d.getLinkDiameter()

teste = {
    "id":d.getLinkNameID(),
    "startnode":startnode,
    "endnode":endnode,
    "roughness":roughness,
    "length":length,
    "diameter":diameter,

}
df_link = pd.DataFrame(teste)

df_link

# nó alvo
no_alvo = "5589"

# filtrar links onde o nó final é o alvo
df_links_com_no_inicial = df_link[df_link['startnode'] == no_alvo]

# pegar só os IDs dessas redes
ids_links_com_no_inicial = df_links_com_no_inicial['id'].tolist()

# Pegar só o primeiro:
primeiro_id = ids_links_com_no_inicial[0] if ids_links_com_no_inicial else None

print("Primeiro ID:", primeiro_id)

# print(ids_links_com_no_inicial)


Primeiro ID: 1408


In [80]:
teste = df_link[df_link['startnode']=='11160']
teste

,id,startnode,endnode,roughness,length,diameter
12416,VRP.VCP.NOVA01,11160,937,0.0,0.0,100.0


In [72]:
#Criando dicionário dos IDs das redes
dict_id_name_to_index = {nome:index for nome, index in zip(d.getLinkPipeNameID(), d.getLinkIndex())}
dict_id_name_to_index

print(d.getLinkVelocity(dict_id_name_to_index['11884']))
print(d.getLinkFlows(dict_id_name_to_index['11884']))
print(d.getLinkHeadloss(dict_id_name_to_index['11884']))

0.1750430017709732
-0.5456549525260925
0.015450065024197102


In [73]:
import datetime
import pandas as pd

# Nome ou ID do link alvo
link_alvo = '11884'
index_link = dict_id_name_to_index[link_alvo]

# Reabre análise hidráulica
d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()

# Listas para armazenar valores hora a hora
velocidade_horaria = []
vazao_horaria = []
headloss_horaria = []
horas = []

tstep = 1
hora = 0

while tstep > 0:
    t = d.runHydraulicAnalysis()
    
    # Captura valores do link específico nesse instante
    velocidade = d.getLinkVelocity()[index_link]
    vazao = abs(d.getLinkFlows()[index_link])
    headloss = d.getLinkHeadloss()[index_link]

    # Guarda
    velocidade_horaria.append(velocidade)
    vazao_horaria.append(vazao)
    headloss_horaria.append(headloss)

    # Guarda hora formatada
    tempo_formatado = str(datetime.timedelta(hours=hora))
    horas.append(tempo_formatado)

    # Avança
    tstep = d.nextHydraulicAnalysisStep()
    hora += 1

d.closeHydraulicAnalysis()

# Monta DataFrame
df_link_horario = pd.DataFrame({
    "Hora": horas,
    "Velocidade_m_s": velocidade_horaria,
    "Vazao_L_s": vazao_horaria,
    "Perda_carga_m": headloss_horaria
})

print(df_link_horario)


              Hora  Velocidade_m_s  Vazao_L_s  Perda_carga_m
0          0:00:00        0.175043   0.545655       0.017169
1          1:00:00        0.148272   0.462202       0.012626
2          2:00:00        0.131797   0.410846       0.010151
3          3:00:00        0.121500   0.378749       0.008732
4          4:00:00        0.117382   0.365910       0.008191
5          5:00:00        0.117382   0.365910       0.008191
6          6:00:00        0.127678   0.398007       0.009572
7          7:00:00        0.168865   0.526397       0.016064
8          8:00:00        0.197696   0.616269       0.021510
9          9:00:00        0.230645   0.718981       0.028617
10        10:00:00        0.253298   0.789595       0.034039
11        11:00:00        0.267713   0.834531       0.037713
12        12:00:00        0.271831   0.847370       0.038795
13        13:00:00        0.263594   0.821692       0.036646
14        14:00:00        0.253298   0.789595       0.034039
15        15:00:00      

In [22]:
lista = d.getLinkValveNameID()

# Ordenar em ordem alfabética
lista_ordenada = sorted(lista)

print("Quantidade de itens:", len(lista_ordenada))

# Printar cada item
for item in lista_ordenada:
    print(f"'{item}',")

Quantidade de itens: 53
'FCV',
'VRP.SAM.001',
'VRP.SAM.002',
'VRP.SAM.003',
'VRP.SAM.004',
'VRP.SAM.005',
'VRP.SAM.006',
'VRP.SAM.007',
'VRP.SAM.008',
'VRP.SAM.009',
'VRP.SAM.011',
'VRP.SAM.012',
'VRP.SAM.013',
'VRP.SAM.014',
'VRP.SAM.015',
'VRP.SAM.016',
'VRP.SAM.017',
'VRP.SAM.018R',
'VRP.SAM.019',
'VRP.SAM.020',
'VRP.SAM.021',
'VRP.SAM.022',
'VRP.SAM.023',
'VRP.SAM.024',
'VRP.SAM.025',
'VRP.SAM.026',
'VRP.SAM.027',
'VRP.SAM.028',
'VRP.SAM.029',
'VRP.SAM.030',
'VRP.SAM.NOVA.001',
'VRP.SAM.NOVA.002',
'VRP.SAM.NOVA.003',
'VRP.SAM.NOVA.004',
'VRP.SAM.NOVA.005',
'VRP.SAM.NOVA.006',
'VRP.SAM.NOVA.007',
'VRP.SAM.NOVA.008',
'VRP.SAM.NOVA.009',
'VRP.SAM.NOVA.010',
'VRP.SAM.NOVA.011',
'VRP.SAM.NOVA.012',
'VRP.SAM.NOVA.013',
'VRP.SAM.NOVA.014',
'VRP.SAM.NOVA.015',
'VRP.SAM.NOVA.016',
'VRP.SAM.NOVA.017',
'VRP.SAM.NOVA.018',
'VRP.SAM.NOVA.019',
'VRP.SAM.NOVA.020',
'VRP.SAM.NOVA.021',
'VRP.SAM.NOVA.022',
'VRP.SMT.NOVA.001',


In [35]:
# ==========================================================
# 📌 ANÁLISE HIDRÁULICA HORA A HORA
# ==========================================================

import datetime
import numpy as np
import pandas as pd

# ==========================================================
# 📌 Função para obter nós conectados à válvula
# ==========================================================
def obter_nos_da_valvula(nome_valvula):

    dict_valve_name_index = {
        nome: index
        for index, nome in enumerate(d.getLinkValveNameID(), start=1)
    }

    valve_id = d.getLinkValveNameID(
        [dict_valve_name_index[nome_valvula]]
    )

    nos_conectados = d.getNodesConnectingLinksID(valve_id)

    if isinstance(nos_conectados, np.ndarray) and nos_conectados.ndim > 1:
        nos_conectados = nos_conectados[0]

    return [str(no) for no in nos_conectados]


# ==========================================================
# 📌 Dicionários auxiliares
# ==========================================================
dict_index_name = {
    index: nome
    for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())
}

dict_name_index = {
    nome: index
    for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())
}

dict_id_name_to_index = {
    id_: idx - 1
    for id_, idx in zip(d.getLinkNameID(), d.getLinkIndex())
}


# ==========================================================
# 📌 Informações da rede
# ==========================================================
dict_ids = {
    index: nome
    for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())
}

startnode = [
    dict_ids[ids[0]]
    for ids in d.getLinkNodesIndex()
]

endnode = [
    dict_ids[ids[1]]
    for ids in d.getLinkNodesIndex()
]

df_link = pd.DataFrame({
    "id": d.getLinkNameID(),
    "startnode": startnode,
    "endnode": endnode,
    "roughness": d.getLinkRoughnessCoeff(),
    "length": d.getLinkLength(),
    "diameter": d.getLinkDiameter()
})


# ==========================================================
# 📌 Lista de válvulas
# ==========================================================
lista_valvulas = [
'VRP.SAM.001',
'VRP.SAM.002',
'VRP.SAM.003',
'VRP.SAM.004',
'VRP.SAM.005',
'VRP.SAM.006',
'VRP.SAM.007',
'VRP.SAM.008',
'VRP.SAM.009',
'VRP.SAM.011',
'VRP.SAM.012',
'VRP.SAM.013',
'VRP.SAM.014',
'VRP.SAM.015',
'VRP.SAM.016',
'VRP.SAM.017',
'VRP.SAM.018R',
'VRP.SAM.019',
'VRP.SAM.020',
'VRP.SAM.021',
'VRP.SAM.022',
'VRP.SAM.023',
'VRP.SAM.024',
'VRP.SAM.025',
'VRP.SAM.026',
'VRP.SAM.027',
'VRP.SAM.028',
'VRP.SAM.029',
'VRP.SAM.030',
'VRP.SAM.NOVA.001',
'VRP.SAM.NOVA.002',
'VRP.SAM.NOVA.003',
'VRP.SAM.NOVA.004',
'VRP.SAM.NOVA.005',
'VRP.SAM.NOVA.006',
'VRP.SAM.NOVA.007',
'VRP.SAM.NOVA.008',
'VRP.SAM.NOVA.009',
'VRP.SAM.NOVA.010',
'VRP.SAM.NOVA.011',
'VRP.SAM.NOVA.012',
'VRP.SAM.NOVA.013',
'VRP.SAM.NOVA.014',
'VRP.SAM.NOVA.015',
'VRP.SAM.NOVA.016',
'VRP.SAM.NOVA.017',
'VRP.SAM.NOVA.018',
'VRP.SAM.NOVA.019',
'VRP.SAM.NOVA.020',
'VRP.SAM.NOVA.021',
'VRP.SAM.NOVA.022',
'VRP.SMT.NOVA.001',
]


# ==========================================================
# 📌 Função para localizar link da válvula
# ==========================================================
def encontrar_link_por_valvula(nome_valvula, df_link, nos_conectados):

    # procura pelo nome da válvula
    df_links_com_valvula = df_link[
        df_link['id'].str.contains(nome_valvula, na=False)
    ]

    if not df_links_com_valvula.empty:
        return df_links_com_valvula.iloc[0]

    # procura pelos nós
    for no in nos_conectados:

        df_links_no = df_link[
            (df_link['startnode'] == no) |
            (df_link['endnode'] == no)
        ]

        if not df_links_no.empty:
            return df_links_no.iloc[0]

    return None


# ==========================================================
# 📌 RODAR SIMULAÇÃO GLOBAL
# ==========================================================
print("▶ Rodando simulação hidráulica...")

d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()

tstep = 1

P = []
tempos_horas = []

while tstep > 0:

    t = d.runHydraulicAnalysis()

    # tempo atual em horas
    hora_atual = int(t / 3600)

    # salva apenas horários inteiros
    if (t % 3600) == 0:

        P.append(d.getNodePressure())

        tempos_horas.append(
            str(datetime.timedelta(hours=hora_atual))
        )

    tstep = d.nextHydraulicAnalysisStep()

d.closeHydraulicAnalysis()

print("✅ Simulação concluída!")
print(f"📌 Total de horários salvos: {len(tempos_horas)}")


# ==========================================================
# 📌 CRIAR EXCEL
# ==========================================================
output_excel = r"Cavitacao\Samambaia\Cav SAM Cenario VRPs novas.xlsx"

with pd.ExcelWriter(output_excel) as writer:

    # ======================================================
    # LOOP DAS VÁLVULAS
    # ======================================================
    for nome_valvula in lista_valvulas:

        print(f"\n➡️ Processando válvula: {nome_valvula}")

        # ==================================================
        # 1️⃣ Nós conectados
        # ==================================================
        nos_conectados = obter_nos_da_valvula(nome_valvula)

        print(f"   🔗 Nós conectados: {nos_conectados}")

        index_nos = [
            dict_name_index[no] - 1
            for no in nos_conectados
        ]

        # ==================================================
        # 2️⃣ PRESSÕES
        # ==================================================
        linhas_pressao = []

        for hora_txt, pressao in zip(tempos_horas, P):

            valores = [
                pressao[idx]
                for idx in index_nos
            ]

            linhas_pressao.append(
                [hora_txt] + valores
            )

        # nomes das colunas
        colunas_pressao = ["Hora"]

        for i in range(len(index_nos)):
            colunas_pressao.append(f"Pressure_{i+1}")

        df_pressao = pd.DataFrame(
            linhas_pressao,
            columns=colunas_pressao
        )

        # ==================================================
        # 3️⃣ LINK ASSOCIADO
        # ==================================================
        link_info = encontrar_link_por_valvula(
            nome_valvula,
            df_link,
            nos_conectados
        )

        # ==================================================
        # 4️⃣ DADOS DO LINK
        # ==================================================
        if link_info is not None:

            id_link = link_info['id']

            comprimento_link = link_info['length']

            index_link = dict_id_name_to_index[id_link]

            print(
                f"   🔍 Link encontrado: "
                f"{id_link} "
                f"({comprimento_link:.2f} m)"
            )

            # ==============================================
            # Rodar simulação do link
            # ==============================================
            d.openHydraulicAnalysis()
            d.initializeHydraulicAnalysis()

            tstep = 1

            horas = []
            velocidades = []
            vazoes = []
            perdas = []

            while tstep > 0:

                t = d.runHydraulicAnalysis()

                hora_atual = int(t / 3600)

                # salva apenas hora cheia
                if (t % 3600) == 0:

                    horas.append(
                        str(datetime.timedelta(hours=hora_atual))
                    )

                    velocidades.append(
                        d.getLinkVelocity()[index_link]
                    )

                    vazoes.append(
                        abs(d.getLinkFlows()[index_link])
                    )

                    perdas.append(
                        d.getLinkHeadloss()[index_link]
                    )

                tstep = d.nextHydraulicAnalysisStep()

            d.closeHydraulicAnalysis()

            # ==============================================
            # DataFrame do link
            # ==============================================
            df_link_horario = pd.DataFrame({
                "Hora": horas,
                "Velocidade_m_s": velocidades,
                "Vazao_L_s": vazoes,
                "Perda_carga_total_m": perdas
            })

            # ==============================================
            # Combinar
            # ==============================================
            df_final = df_pressao.merge(
                df_link_horario,
                on="Hora",
                how="left"
            )

        else:

            print(
                f"   ⚠️ Nenhum link encontrado "
                f"para {nome_valvula}"
            )

            df_final = df_pressao

        # ==================================================
        # 5️⃣ Nome da aba
        # ==================================================
        diametro = (
            str(int(link_info['diameter']))
            if link_info is not None
            else "ND"
        )

        nome_aba = (
            f"{nome_valvula}_{diametro}"
        )[:31]

        # ==================================================
        # 6️⃣ Exportar
        # ==================================================
        df_final.to_excel(
            writer,
            sheet_name=nome_aba,
            index=False
        )

        print(f"   ✅ Aba salva: {nome_aba}")

# ==========================================================
# 📌 FINALIZAÇÃO
# ==========================================================
print("\n✅ Excel criado com sucesso!")
print(f"📁 Arquivo salvo em:\n{output_excel}")

▶ Rodando simulação hidráulica...
✅ Simulação concluída!
📌 Total de horários salvos: 25

➡️ Processando válvula: VRP.SAM.001
   🔗 Nós conectados: ['PM.VRP.SAM.001', '14370']
   🔍 Link encontrado: VRP.SAM.001 (0.00 m)
   ✅ Aba salva: VRP.SAM.001_250

➡️ Processando válvula: VRP.SAM.002
   🔗 Nós conectados: ['PM.VRP.SAM.002', '13276']
   🔍 Link encontrado: VRP.SAM.002 (0.00 m)
   ✅ Aba salva: VRP.SAM.002_300

➡️ Processando válvula: VRP.SAM.003
   🔗 Nós conectados: ['PM.VRP.SAM.003', '13959']
   🔍 Link encontrado: VRP.SAM.003 (0.00 m)
   ✅ Aba salva: VRP.SAM.003_200

➡️ Processando válvula: VRP.SAM.004
   🔗 Nós conectados: ['PM.VRP.SAM.004', '13532']
   🔍 Link encontrado: VRP.SAM.004 (0.00 m)
   ✅ Aba salva: VRP.SAM.004_150

➡️ Processando válvula: VRP.SAM.005
   🔗 Nós conectados: ['PM.VRP.SAM.005', '9004']
   🔍 Link encontrado: VRP.SAM.005 (0.00 m)
   ✅ Aba salva: VRP.SAM.005_300

➡️ Processando válvula: VRP.SAM.006
   🔗 Nós conectados: ['PM.VRP.SAM.006', '14424']
   🔍 Link encontrado: 

In [23]:
import datetime
import numpy as np

# Abrindo a análise hidráulica
d.openHydraulicAnalysis()
d.initializeHydraulicAnalysis()

tstep, P, T_H, D, H, F, S = 1, [], [], [], [], [], []
tempo = []

while tstep > 0:
    t = d.runHydraulicAnalysis()
    P.append(d.getNodePressure())
    D.append(d.getNodeActualDemand())
    H.append(d.getNodeHydraulicHead())
    S.append(d.getLinkStatus())
    F.append(d.getLinkFlows())
    T_H.append(t)
    tstep = d.nextHydraulicAnalysisStep()
    tempo.append(tstep)

d.closeHydraulicAnalysis()

# Dicionários de conversão entre nome e índice
dict_index_name = {index: nome for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
dict_name_index = {nome: index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}

# Nome do nó alvo
nome_no = "4431"
# nome_no = "6132"
index_no = dict_name_index[nome_no] - 1  # índice baseado em 0

# Lista para armazenar as pressões
valores_pressao = []

# Lista para armazenar as pressões
valores_pressao = []

print(f"\nPressões horárias para o nó {nome_no}:\n")
for hora, pressao in enumerate(P):
    tempo_formatado = str(datetime.timedelta(hours=hora))
    valor_pressao = pressao[index_no]
    valores_pressao.append(valor_pressao)
    print(f"{tempo_formatado} - Nó {nome_no}: {valor_pressao:.2f} mca")

# Média geral
media_pressao_geral = np.mean(valores_pressao)
print(f"\n📌 Média GERAL da pressão no nó {nome_no}: {media_pressao_geral:.2f} mca")

# --------- MÉDIA DE UM INTERVALO DE HORAS DEFINIDO PELO USUÁRIO ----------
hora_inicio = 6  # exemplo: 6h
hora_fim = 22   # exemplo: 22h (exclui a hora 22)

pressao_intervalo = {
    i: round(pressao[index_no], 2)
    for i, pressao in enumerate(P)
    if i <= hora_inicio or i >= hora_fim
}

if pressao_intervalo:
    media_intervalo = np.mean(list(pressao_intervalo.values()))
    print(f"📊 Média da pressão no nó {nome_no} entre {hora_inicio:02d}:00 e {hora_fim:02d}:00: {media_intervalo:.2f} mca")
else:
    print(f"⚠️ Nenhum dado de pressão entre {hora_inicio:02d}:00 e {hora_fim:02d}:00.")



Pressões horárias para o nó 4431:

0:00:00 - Nó 4431: 10.00 mca
1:00:00 - Nó 4431: 10.00 mca
2:00:00 - Nó 4431: 10.00 mca
3:00:00 - Nó 4431: 10.00 mca
4:00:00 - Nó 4431: 10.00 mca
5:00:00 - Nó 4431: 10.00 mca
6:00:00 - Nó 4431: 10.00 mca
7:00:00 - Nó 4431: 10.00 mca
8:00:00 - Nó 4431: 10.00 mca
9:00:00 - Nó 4431: 10.00 mca
10:00:00 - Nó 4431: 10.00 mca
11:00:00 - Nó 4431: 10.00 mca
12:00:00 - Nó 4431: 10.00 mca
13:00:00 - Nó 4431: 10.00 mca
14:00:00 - Nó 4431: 10.00 mca
15:00:00 - Nó 4431: 10.00 mca
16:00:00 - Nó 4431: 10.00 mca
17:00:00 - Nó 4431: 10.00 mca
18:00:00 - Nó 4431: 10.00 mca
19:00:00 - Nó 4431: 10.00 mca
20:00:00 - Nó 4431: 10.00 mca
21:00:00 - Nó 4431: 10.00 mca
22:00:00 - Nó 4431: 10.00 mca
23:00:00 - Nó 4431: 10.00 mca
1 day, 0:00:00 - Nó 4431: 10.00 mca

📌 Média GERAL da pressão no nó 4431: 10.00 mca
📊 Média da pressão no nó 4431 entre 06:00 e 22:00: 10.00 mca


In [291]:
pressao_intervalo

{0: 10.0,
 1: 10.0,
 2: 10.0,
 3: 10.0,
 4: 10.0,
 5: 10.0,
 6: 10.0,
 22: 10.0,
 23: 10.0,
 24: 10.0}

In [323]:
if nome_nó in ids_nós:
    indice_nó = ids_nós[nome_nó]
    print(f"Pressão horária do nó {nome_nó}:")

Pressão horária do nó 553:


In [324]:
d.getNodePressure(ids_nós['553'])

33.780426025390625

# Criação de arquivos Excel a partir do Epanet

In [6]:
import pandas as pd
list_ids = d.getNodeNameID()
eleva = d.getNodeElevations()

cordenada = d.getNodeCoordinates()
x = cordenada["x"]
y = cordenada["y"]

base =  d.getNodeBaseDemands()

teste = {
    "id":list_ids,
    "Elevação":eleva,
    "Demanda":list(base[1]),
    "X_COORD":x.values(),
    "Y_COORD":y.values(),

}
df_node = pd.DataFrame(teste)
df_node

,id,Elevação,Demanda,X_COORD,Y_COORD
0,0,1026.199951,0.077999,184364.028,8245057.145
1,1,1026.199951,0.064305,184365.120,8245055.470
2,2,1026.199951,0.032153,184365.848,8245054.355
3,3,1035.920044,0.000000,184122.502,8245560.306
4,4,1036.339966,0.000000,184125.377,8245557.215
...,...,...,...,...,...
2844,2885,1023.229980,0.000000,182405.585,8243721.358
2845,2886,1023.229980,0.000000,182404.824,8243722.642
2846,2887,1022.099976,0.000000,182405.434,8243752.526
2847,105,1041.599976,0.000000,181085.629,8242612.082


In [73]:
dict_ids = {index:nome for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}

roughness = d.getLinkRoughnessCoeff()

startnode = [dict_ids[ids[0]] for ids in d.getLinkNodesIndex()]
endnode = [dict_ids[ids[1]] for ids in d.getLinkNodesIndex()]
roughness = d.getLinkRoughnessCoeff()
length = d.getLinkLength()
diameter = d.getLinkDiameter()

teste = {
    "id":d.getLinkNameID(),
    "startnode":startnode,
    "endnode":endnode,
    "roughness":roughness,
    "length":length,
    "diameter":diameter,

}
df_link = pd.DataFrame(teste)

df_link

,id,startnode,endnode,roughness,length,diameter
0,0,7867,7504,140.0,3.271127,110.0
1,1,7664,7665,135.0,4.583378,250.0
2,2,7663,7664,135.0,11.899364,250.0
3,3,7662,7663,135.0,6.460561,250.0
4,4,7661,7662,135.0,6.871102,250.0
...,...,...,...,...,...,...
7952,VRP.nao,6938,PM.VRP.SSB.011,0.0,0.000000,300.0
7953,VRP.SSB.014,PM.VRP.SSB.014,7332,0.0,0.000000,280.0
7954,VRP.SSB.018,5716,5715,0.0,0.000000,90.0
7955,VRP.SSB.015,PM.VRP.SSB.015,154,0.0,0.000000,200.0


# Tratamento para Nós

## Abrindo arquivos

In [6]:
uniforme = pd.read_excel('Tabelas para calibração\\Vicente\\Demanda Final Ponderada.xlsx')
uniforme

,Unnamed: 0,Input_FID,Join_Count_x,Idade (anos),Consumo,Join_Count_y,NODENUM_x,DIAMETRO,CHW,perda_x,...,material,rugosidade,diameter,Xinicial,Yinicial,Xfinal,Yfinal,perda_y,NOME,Demanda
0,0,0,NaN,NaN,NaN,2,0,51.4,137.5,0.014627,...,PEAD,137.5,63,174884.3920,8.248748e+06,174888.6289,8.248810e+06,0.014627,VRP.VCP.024,0.000000
1,1,1,NaN,NaN,NaN,2,1,51.4,137.5,0.014627,...,PEAD,137.5,63,174884.3920,8.248748e+06,174888.6289,8.248810e+06,0.014627,VRP.VCP.024,0.000000
2,2,2,NaN,NaN,NaN,0,2,51.4,137.5,0.014627,...,PEAD,137.5,63,174882.9589,8.248814e+06,174878.2089,8.248775e+06,0.014627,VRP.VCP.024,0.000000
3,3,3,12.0,17.421918,0.010293,12,3,51.4,137.5,0.014627,...,PEAD,137.5,63,174882.9589,8.248814e+06,174878.2089,8.248775e+06,0.014627,VRP.VCP.024,0.123201
4,4,4,0.0,17.506849,0.001146,0,4,32.6,137.5,0.014627,...,PEAD,137.5,40,175051.9296,8.249256e+06,174947.8891,8.249256e+06,0.014627,VRP.VCP.030,0.028580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11643,11643,11747,1.0,3.162100,0.033627,1,11747,51.4,140.0,0.014627,...,PEAD,0.0,63,174554.8988,8.249437e+06,174553.3817,8.249543e+06,0.014627,VRP.VCP.030,0.076005
11644,11644,11748,NaN,NaN,NaN,0,11748,147.2,137.5,0.014627,...,PEAD,137.5,180,177403.7319,8.251832e+06,177412.9791,8.251832e+06,0.014627,VRP.VCP.012,0.000000
11645,11645,11749,NaN,NaN,NaN,0,11749,51.4,137.5,0.014627,...,PEAD,137.5,63,177447.1935,8.251231e+06,177443.2278,8.251231e+06,0.014627,VRP.VCP.039,0.000000
11646,11646,11750,NaN,NaN,NaN,0,11750,147.2,137.5,0.014627,...,PEAD,137.5,180,177614.0103,8.252295e+06,177624.5050,8.252297e+06,0.014627,VRP.VCP.012,0.000000


In [5]:
# #Abrindo arquivo dos nós que serão calibrados
# nos = pd.read_excel("Tabelas para calibração\\Nós_calibração.xls")
# nos

In [348]:
2940 in nos["FID"].tolist()

True

In [405]:
nos = nos[['FID']]

In [811]:
contagem = pd.read_excel("Tabelas para calibração\Thiessen Conserto.xls")
# contagem = contagem[contagem['UDA']== 'UDA.GAM.003']
contagem.columns

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\1007078809.py:1: SyntaxWarning: invalid escape sequence '\T'
  contagem = pd.read_excel("Tabelas para calibração\Thiessen Conserto.xls")


Index(['FID', 'Join_Count', 'TARGET_FID', 'Id', 'Input_FID', 'ASSETGROUP',
       'ASSETTYPE', 'FROMDEVICE', 'TODEVICETE', 'GLOBALID', 'creationda',
       'creator', 'lastupdate', 'updatedby', 'installdat', 'notes', 'diameter',
       'lifecycles', 'inserviced', 'retireddat', 'material', 'designtype',
       'codunidade', 'posicionam', 'contratoob', 'tiposistem', 'tipodesenh',
       'rugosidade', 'dataimplan', 'Shape__Len', 'ORIG_FID', 'ORIG_SEQ',
       'Xinicial', 'Yinicial', 'Xfinal', 'Yfinal', 'UDA', 'Cota', 'POINT_X',
       'POINT_Y', 'POINT_Z', 'POINT_M', 'Data_da_ab', 'Localidade',
       'F_OS__UDAs', 'DMC', 'Tipo_de_se', 'Código_e_', 'OS', 'LATITUDE',
       'LONGITUDE'],
      dtype='object')

In [734]:
teste = contagem[contagem["Input_FID"]=='2591']
teste

,FID,Join_Count,TARGET_FID,Id,Input_FID,ASSETGROUP,ASSETTYPE,FROMDEVICE,TODEVICETE,GLOBALID,...,POINT_M,Data_da_ab,Localidade,F_OS__UDAs,DMC,Tipo_de_se,Código_e_,OS,LATITUDE,LONGITUDE


In [7]:
lista_nos = uniforme[['NODENUM_x','Consumo']]

In [407]:
nos = pd.merge(nos,todos_nos, left_on="FID",right_on="Ligacao_thiessen.Input_FID",how='left')

In [408]:
nos.columns

Index(['FID', 'Unnamed: 0', 'Ligacao_thiessen.Input_FID',
       'Ligacao_thiessen.Consumo_m', 'perda', 'Demanda+perda'],
      dtype='object')

In [409]:
nos['Ligacao_thiessen.Consumo_m'].sum()

14.14

In [812]:
nos = pd.merge(todos_nos,contagem,left_on="Ligacao_thiessen.Input_FID",right_on="Input_FID",how='right')

In [813]:
nos

,Unnamed: 0,Ligacao_thiessen.Input_FID,Ligacao_thiessen.Consumo_m,perda,Demanda+perda,FID,Join_Count,TARGET_FID,Id,Input_FID,...,POINT_M,Data_da_ab,Localidade,F_OS__UDAs,DMC,Tipo_de_se,Código_e_,OS,LATITUDE,LONGITUDE
0,NaN,NaN,NaN,NaN,NaN,0,0,0,0,2954,...,NaN,NaT,,,,,,,0.000000,0.000000
1,NaN,NaN,NaN,NaN,NaN,1,0,1,0,1325,...,NaN,NaT,,,,,,,0.000000,0.000000
2,NaN,NaN,NaN,NaN,NaN,2,0,2,0,1327,...,NaN,NaT,,,,,,,0.000000,0.000000
3,NaN,NaN,NaN,NaN,NaN,3,0,3,0,1326,...,NaN,NaT,,,,,,,0.000000,0.000000
4,NaN,NaN,NaN,NaN,NaN,4,0,4,0,2957,...,NaN,NaT,,,,,,,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3305,2.0,3.0,0.24,0.005579,0.245579,3305,0,3305,0,3,...,NaN,NaT,,,,,,,0.000000,0.000000
3306,660.0,1088.0,0.31,0.005579,0.315579,3306,1,3306,0,1088,...,NaN,2024-04-05,Gama,UDA.GAM.002,Não informada,RAMAL,(8101008013015) - Conserto em tubulação de águ...,2405610042490061,-16.013549,-48.061575
3307,NaN,NaN,NaN,NaN,NaN,3307,0,3307,0,2,...,NaN,NaT,,,,,,,0.000000,0.000000
3308,1468.0,2393.0,0.03,0.026565,0.056565,3308,0,3308,0,2393,...,NaN,NaT,,,,,,,0.000000,0.000000


In [814]:
nos

,Unnamed: 0,Ligacao_thiessen.Input_FID,Ligacao_thiessen.Consumo_m,perda,Demanda+perda,FID,Join_Count,TARGET_FID,Id,Input_FID,...,POINT_M,Data_da_ab,Localidade,F_OS__UDAs,DMC,Tipo_de_se,Código_e_,OS,LATITUDE,LONGITUDE
0,NaN,NaN,NaN,NaN,NaN,0,0,0,0,2954,...,NaN,NaT,,,,,,,0.000000,0.000000
1,NaN,NaN,NaN,NaN,NaN,1,0,1,0,1325,...,NaN,NaT,,,,,,,0.000000,0.000000
2,NaN,NaN,NaN,NaN,NaN,2,0,2,0,1327,...,NaN,NaT,,,,,,,0.000000,0.000000
3,NaN,NaN,NaN,NaN,NaN,3,0,3,0,1326,...,NaN,NaT,,,,,,,0.000000,0.000000
4,NaN,NaN,NaN,NaN,NaN,4,0,4,0,2957,...,NaN,NaT,,,,,,,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3305,2.0,3.0,0.24,0.005579,0.245579,3305,0,3305,0,3,...,NaN,NaT,,,,,,,0.000000,0.000000
3306,660.0,1088.0,0.31,0.005579,0.315579,3306,1,3306,0,1088,...,NaN,2024-04-05,Gama,UDA.GAM.002,Não informada,RAMAL,(8101008013015) - Conserto em tubulação de águ...,2405610042490061,-16.013549,-48.061575
3307,NaN,NaN,NaN,NaN,NaN,3307,0,3307,0,2,...,NaN,NaT,,,,,,,0.000000,0.000000
3308,1468.0,2393.0,0.03,0.026565,0.056565,3308,0,3308,0,2393,...,NaN,NaT,,,,,,,0.000000,0.000000


In [815]:
nos.columns

Index(['Unnamed: 0', 'Ligacao_thiessen.Input_FID',
       'Ligacao_thiessen.Consumo_m', 'perda', 'Demanda+perda', 'FID',
       'Join_Count', 'TARGET_FID', 'Id', 'Input_FID', 'ASSETGROUP',
       'ASSETTYPE', 'FROMDEVICE', 'TODEVICETE', 'GLOBALID', 'creationda',
       'creator', 'lastupdate', 'updatedby', 'installdat', 'notes', 'diameter',
       'lifecycles', 'inserviced', 'retireddat', 'material', 'designtype',
       'codunidade', 'posicionam', 'contratoob', 'tiposistem', 'tipodesenh',
       'rugosidade', 'dataimplan', 'Shape__Len', 'ORIG_FID', 'ORIG_SEQ',
       'Xinicial', 'Yinicial', 'Xfinal', 'Yfinal', 'UDA', 'Cota', 'POINT_X',
       'POINT_Y', 'POINT_Z', 'POINT_M', 'Data_da_ab', 'Localidade',
       'F_OS__UDAs', 'DMC', 'Tipo_de_se', 'Código_e_', 'OS', 'LATITUDE',
       'LONGITUDE'],
      dtype='object')

In [816]:
nos = nos[["Ligacao_thiessen.Input_FID",'Input_FID',"Ligacao_thiessen.Consumo_m","Join_Count","UDA"]]

In [824]:
UDA3 = nos.copy()
UDA3 = UDA3[UDA3['UDA']=='UDA.GAM.003']
# UDA3 = UDA3['Ligacao_thiessen.Consumo_m'].fillna(0)

In [825]:
UDA3['Ligacao_thiessen.Consumo_m'].sum()

33.67

In [826]:
UDA3['Ligacao_thiessen.Consumo_m'].fillna(0,inplace=True)

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\1408711143.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  UDA3['Ligacao_thiessen.Consumo_m'].fillna(0,inplace=True)


In [827]:
UDA3['perda'] = (71.096013 - (UDA3['Ligacao_thiessen.Consumo_m'].sum())) / len(UDA3)

In [832]:
UDA3['perda'].sum()

37.426013

In [828]:
UDA3['Demanda_final'] = UDA3['Ligacao_thiessen.Consumo_m'] + UDA3['perda']

In [829]:
UDA4 = nos.copy()
UDA4 = UDA4[UDA4['UDA']=='UDA.GAM.004']

In [830]:
UDA4['Ligacao_thiessen.Consumo_m'].sum()

18.150000000000002

In [831]:
UDA4['Ligacao_thiessen.Consumo_m'].fillna(0,inplace=True)

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\1313366631.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  UDA4['Ligacao_thiessen.Consumo_m'].fillna(0,inplace=True)


In [843]:
UDA4['perda'] = (38.6418 - (UDA4['Ligacao_thiessen.Consumo_m'].sum())) / len(UDA4)
UDA4['perda'].sum()

20.491800000000005

In [834]:
UDA4['Demanda_final'] = UDA4['Ligacao_thiessen.Consumo_m'] + UDA4['perda']

In [837]:
UDA2 = nos.copy()
UDA2 = UDA2[UDA2['UDA']=='UDA.GAM.002']

In [840]:
UDA2['Ligacao_thiessen.Consumo_m'].sum()

63.38

In [839]:
UDA2['Ligacao_thiessen.Consumo_m'].fillna(0,inplace=True)

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\4206468859.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  UDA2['Ligacao_thiessen.Consumo_m'].fillna(0,inplace=True)


In [842]:
UDA2['perda'] = (115.4643 - (UDA2['Ligacao_thiessen.Consumo_m'].sum())) / len(UDA2)
UDA2['perda'].sum()

52.084299999999985

In [844]:
UDA2['Demanda_final'] = UDA2['Ligacao_thiessen.Consumo_m'] + UDA2['perda']

In [846]:
df_final = pd.concat([UDA2,UDA3,UDA4])

In [848]:
df_final['Demanda_final'].sum()

225.202113

In [550]:
nos.to_excel('Novo calculo das perdas.xlsx')

In [416]:
nos['perda'] = ( 32.08019 - (nos['Ligacao_thiessen.Consumo_m'].sum())) / len(nos)

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\2427826448.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nos['perda'] = ( 32.08019 - (nos['Ligacao_thiessen.Consumo_m'].sum())) / len(nos)


In [417]:
df = nos.copy()

In [745]:
df = pd.read_excel('Tabelas para calibração\\teste.xlsx')

In [854]:
df_final = df_final.rename(columns={"Input_FID": "NODENUM"})

In [855]:
df_final

,NODENUM,Join_Count,Ligacao_thiessen.Consumo_m,perda
0,2954,0,0.00,0.025784
1,1325,0,0.00,0.025784
2,1327,0,0.00,0.025784
3,1326,0,0.00,0.025784
4,2957,0,0.00,0.025784
...,...,...,...,...
3188,1618,0,0.00,0.045843
3246,1439,0,0.04,0.045843
3249,1024,4,0.09,0.045843
3294,2227,0,0.00,0.045843


In [739]:
2940 in df["NODENUM"].tolist()

True

In [856]:
df_final = df_final[['NODENUM','Join_Count','Ligacao_thiessen.Consumo_m',"perda"]]

In [857]:
df_final

,NODENUM,Join_Count,Ligacao_thiessen.Consumo_m,perda
0,2954,0,0.00,0.025784
1,1325,0,0.00,0.025784
2,1327,0,0.00,0.025784
3,1326,0,0.00,0.025784
4,2957,0,0.00,0.025784
...,...,...,...,...
3188,1618,0,0.00,0.045843
3246,1439,0,0.04,0.045843
3249,1024,4,0.09,0.045843
3294,2227,0,0.00,0.045843


In [ ]:
ligacao = pd.read_excel('Tabelas para calibração\Ligacao_gama.xlsx')
ligacao = ligacao[['Ligacao_thiessen.Input_FID','Ligacao_thiessen.Consumo_m']]
ligacao = pd.merge(ligacao,df,left_on="Ligacao_thiessen.Input_FID",right_on="NODENUM",how='inner')

<>:1: SyntaxWarning: invalid escape sequence '\L'
<>:1: SyntaxWarning: invalid escape sequence '\L'
C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\1280179460.py:1: SyntaxWarning: invalid escape sequence '\L'
  ligacao = pd.read_excel('Tabelas para calibração\Ligacao_gama.xlsx')


In [ ]:
ligacao.columns

Index(['Ligacao_thiessen.Input_FID', 'Ligacao_thiessen.Consumo_m', 'NODENUM',
       'Join_Count'],
      dtype='object')

In [859]:
df = df_final

In [880]:
df1 = UDA2
df2 = UDA3
df3 = UDA4

In [882]:
df1 = df1.rename(columns={"Input_FID": "NODENUM"})
df2 = df2.rename(columns={"Input_FID": "NODENUM"})
df3 = df3.rename(columns={"Input_FID": "NODENUM"})

In [883]:
dic_df = {"df1": df1, "df2": df2,"df3": df3}

In [ ]:
import numpy as np
import pandas as pd

def ajustar_perdas_com_pesos(perda_total, dic_df):
    # Agrupar os dados por nó e somar as perdas e o número de serviços
    perdas_por_no = dic_df.groupby('NODENUM')['perda'].sum().to_dict()
    servicos_por_no = dic_df.groupby('NODENUM')['Join_Count'].sum().to_dict()

    # Garantir que não há serviços negativos
    servicos_por_no = np.array([servicos_por_no.get(no, 0) for no in perdas_por_no.keys()])
    
    # Criar pesos adaptativos com base na quantidade de serviços
    min_servicos = np.min(servicos_por_no)
    max_servicos = np.max(servicos_por_no)

    if min_servicos == max_servicos:
        # Se todos os nós têm a mesma quantidade de serviços, distribuir uniformemente
        pesos = np.ones_like(servicos_por_no)
    else:
        # Normalizar pesos entre 0.5 e 1.5 proporcionalmente aos serviços
        pesos = (servicos_por_no - min_servicos) / (max_servicos - min_servicos)

    # Ajustar as perdas para cada nó com base nas ligações
    perdas_ajustadas = np.zeros_like(servicos_por_no, dtype=float)

    for i, no in enumerate(perdas_por_no.keys()):
        perdas_ajustadas[i] = perdas_por_no[no] * pesos[i]

    # Normalizar as perdas para não ultrapassar a perda total
    perdas_ajustadas = perdas_ajustadas * (perda_total / np.sum(perdas_ajustadas))

    # Criar um dicionário para associar as perdas ajustadas aos nós
    perdas_ajustadas_dict = dict(zip(perdas_por_no.keys(), perdas_ajustadas))

    # Adicionar a coluna 'perda_ajustada' no DataFrame
    dic_df['perda_ajustada'] = dic_df['NODENUM'].map(perdas_ajustadas_dict)

    return dic_df


# df = pd.DataFrame(data)

perda_total = df['perda'].sum()
df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df)

In [885]:
# df = pd.DataFrame(data)

perda_total = df1['perda'].sum()
df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df1)

In [886]:
# df = pd.DataFrame(data)

perda_total = df2['perda'].sum()
df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df2)

In [887]:
# df = pd.DataFrame(data)

perda_total = df3['perda'].sum()
df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df3)

In [889]:
df = pd.concat([df1,df2,df3])

In [ ]:
# import numpy as np
# import pandas as pd

# def ajustar_perdas_com_pesos(perda_total, df):
#     # Agrupar os dados por nó e somar as perdas e o número de serviços
#     perdas_por_no = df.groupby('NODENUM')['perda'].sum().to_dict()
#     servicos_por_no = df.groupby('NODENUM')['Join_Count'].sum().to_dict()

#     # Garantir que não há serviços negativos
#     servicos_por_no = np.array([servicos_por_no.get(no, 0) for no in perdas_por_no.keys()])
    
#     # Criar pesos adaptativos com base na quantidade de serviços
#     min_servicos = np.min(servicos_por_no)
#     max_servicos = np.max(servicos_por_no)

#     if min_servicos == max_servicos:
#         # Se todos os nós têm a mesma quantidade de serviços, distribuir uniformemente
#         pesos = np.ones_like(servicos_por_no)
#     else:
#         # Normalizar pesos entre 0.5 e 1.5 proporcionalmente aos serviços
#         pesos = (servicos_por_no - min_servicos) / (max_servicos - min_servicos)

#     # Ajustar as perdas para cada nó com base nas ligações
#     perdas_ajustadas = np.zeros_like(servicos_por_no, dtype=float)

#     for i, no in enumerate(perdas_por_no.keys()):
#         perdas_ajustadas[i] = perdas_por_no[no] * pesos[i]

#     # Normalizar as perdas para não ultrapassar a perda total
#     perdas_ajustadas = perdas_ajustadas * (perda_total / np.sum(perdas_ajustadas))

#     # Criar um dicionário para associar as perdas ajustadas aos nós
#     perdas_ajustadas_dict = dict(zip(perdas_por_no.keys(), perdas_ajustadas))

#     # Adicionar a coluna 'perda_ajustada' no DataFrame
#     df['perda_ajustada'] = df['NODENUM'].map(perdas_ajustadas_dict)

#     return df


# # df = pd.DataFrame(data)

# perda_total = df['perda'].sum()
# df_com_perda_ajustada = ajustar_perdas_com_pesos(perda_total, df)

In [890]:
df

,Ligacao_thiessen.Input_FID,NODENUM,Ligacao_thiessen.Consumo_m,Join_Count,UDA,perda,Demanda_final,perda_ajustada
0,NaN,2954,0.00,0,UDA.GAM.002,0.025784,0.025784,0.000000
1,NaN,1325,0.00,0,UDA.GAM.002,0.025784,0.025784,0.000000
2,NaN,1327,0.00,0,UDA.GAM.002,0.025784,0.025784,0.000000
3,NaN,1326,0.00,0,UDA.GAM.002,0.025784,0.025784,0.000000
4,NaN,2957,0.00,0,UDA.GAM.002,0.025784,0.025784,0.000000
...,...,...,...,...,...,...,...,...
3188,NaN,1618,0.00,0,UDA.GAM.004,0.045843,0.045843,0.000000
3246,1439.0,1439,0.04,0,UDA.GAM.004,0.045843,0.085843,0.000000
3249,1024.0,1024,0.09,4,UDA.GAM.004,0.045843,0.135843,0.139637
3294,NaN,2227,0.00,0,UDA.GAM.004,0.045843,0.045843,0.000000


In [755]:
df['Ligacao_thiessen.Consumo_m'].fillna(0, inplace=True)

C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\1422320382.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Ligacao_thiessen.Consumo_m'].fillna(0, inplace=True)


In [891]:
df['Demanda_final']=df['Ligacao_thiessen.Consumo_m']+df['perda_ajustada']

In [892]:
df[['perda','Join_Count','perda_ajustada']].value_counts().reset_index()

,perda,Join_Count,perda_ajustada,count
0,0.025784,0,0.000000,1415
1,0.044396,0,0.000000,432
2,0.025784,1,0.038724,297
3,0.045843,0,0.000000,229
4,0.044396,1,0.031085,162
5,0.025784,2,0.077449,139
6,0.045843,1,0.034909,98
7,0.044396,2,0.062169,86
8,0.025784,3,0.116173,68
9,0.044396,3,0.093254,48


In [ ]:
d.getLinkPipeIndex

In [ ]:
dict_link_name_index = {nome:index for nome, index in zip(d.getLinkPipeNameID(), d.getLinkPipeIndex())}
d.getLinkRoughnessCoeff(dict_link_name_index[''])

SyntaxError: invalid syntax. Perhaps you forgot a comma? (661190924.py, line 2)

In [893]:
df['Demanda_final'].sum()

225.202113

In [578]:
resultado = df[df['NODENUM'].isin([2940])]
resultado

,NODENUM,Join_Count,Ligacao_thiessen.Consumo_m,perda,perda_ajustada,Demanda_final
1592,2940,1,0.0,0.025782,0.017813,0.017813


In [866]:
df['Demanda_final'].sum()

225.20211299999997

In [778]:
sort = df.sort_values(by="NODENUM")
sort

,NODENUM,Input_FID,Join_Count,Ligacao_thiessen.Consumo_m,perda,perda_ajustada,Demanda_final
103,9,9,0,0.00,0.044396,0.000000,0.000000
120,10,10,2,0.05,0.044396,0.062169,0.112169
815,11,11,5,0.10,0.044396,0.155424,0.255424
819,12,12,4,0.11,0.044396,0.124339,0.234339
742,13,13,0,0.06,0.044396,0.000000,0.060000
...,...,...,...,...,...,...,...
272,3583,3583,1,0.00,0.044396,0.031085,0.031085
253,3584,3584,0,0.00,0.044396,0.000000,0.000000
265,3585,3585,0,0.00,0.044396,0.000000,0.000000
273,3586,3586,0,0.00,0.044396,0.000000,0.000000


In [779]:
print(sort['Demanda_final'].unique())

[0.         0.11216946 0.25542364 0.23433891 0.06       0.08
 0.03108473 0.08108473 0.07108473 0.09216946 0.05       0.02
 0.20325419 0.25216946 0.3875931  0.54650837 0.2475931  0.12433891
 0.06108473 0.19433891 0.04108473 0.65627093 0.59301674 0.15108473
 0.01       0.15       0.06216946 0.10108473 0.07       0.3175931
 0.13433891 0.05108473 0.09108473 0.10216946 0.28650837 0.13216946
 0.26650837 0.18108473 0.12216946 0.42867783 0.04       0.12108473
 0.12       0.28867783 0.27650837 0.14325419 0.15325419 0.19542364
 0.28433891 0.11108473 0.18216946 0.5351862  0.49627093 0.2775931
 0.20433891 0.07216946 0.22216946 0.08216946 0.17108473 0.23108473
 0.03       0.33650837 0.26542364 0.42084728 0.32650837 0.14216946
 0.23325419 0.32976256 0.25650837 0.21216946 0.16216946 0.32216946
 0.22542364 0.32433891 0.16       0.38542364 0.13325419 0.57410147
 0.09325419 0.16433891 0.19108473 0.21433891 0.11325419 0.09
 0.24433891 0.28325419 0.21650837 0.39084728 0.18433891 0.17325419
 0.20650837 0.2

In [780]:
sort['NODENUM'] = pd.to_numeric(sort['NODENUM'], errors='coerce')


In [781]:
2940 in sort['NODENUM'].tolist()

False

In [785]:
resultado = sort[sort['NODENUM']==1554]
resultado

,NODENUM,Input_FID,Join_Count,Ligacao_thiessen.Consumo_m,perda,perda_ajustada,Demanda_final
2,1554,1554,0,0.03,0.044396,0.0,0.03


In [786]:
entao = df[df['NODENUM']==1558]
entao

,NODENUM,Input_FID,Join_Count,Ligacao_thiessen.Consumo_m,perda,perda_ajustada,Demanda_final
19,1558,1558,2,0.13,0.044396,0.062169,0.192169


In [ ]:
df.to_excel('Tabelas para calibração\demanda dos nós.xlsx')

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_27852\1499859806.py:1: SyntaxWarning: invalid escape sequence '\d'
  df.to_excel('Tabelas para calibração\demanda dos nós.xlsx')


In [ ]:
lista_nos = df[['NODENUM', 'Demanda_final']].values.tolist()


In [797]:
df.to_excel('Tabelas para calibração\\Lista UDA 003.xlsx')

In [894]:
#Criando uma lista com o ID dos Nós que serão calibrados

list_NODENUM = df['NODENUM'].values.tolist()
list_Demanda_final = df['Demanda_final'].values.tolist()

## Tratando Epanet

In [ ]:
current_demanda = list(d.getNodeBaseDemands(index).values())[0]
current_demanda

array([0.])

In [ ]:
count_item

651

In [ ]:
d.getNodeBaseDemands()[1]

array([5.006423  , 2.2647059 , 2.12273812, ..., 0.        , 0.        ,
       0.        ])

# Alterando a demanda dos Nós

## Alterar só os valores

In [4]:
import pandas as pd
list_ids = d.getNodeNameID()
eleva = d.getNodeElevations()
pattern_dict = d.getNodeDemandPatternNameID() 

cordenada = d.getNodeCoordinates()
x = cordenada["x"]
y = cordenada["y"]

base =  d.getNodeBaseDemands()
# pattern_list = list(pattern_dict.values())[0] #Aqui mudar quando tiver padrão de consumo

teste = {
    "id":list_ids,
    "Elevação":eleva,
    "Demanda":list(base[1]),
    "X_COORD":x.values(),
    "Y_COORD":y.values(),
    # "Pattern":pattern_list
}
df_node = pd.DataFrame(teste)
df_node["id"] = df_node["id"].astype(str)
# Substituir NaN por string vazia
# df_node["Pattern"] = [
#     "" if pd.isna(v) else v
#     for v in df_node["Pattern"].values
# ]

df_node

,id,Elevação,Demanda,X_COORD,Y_COORD
0,0,1055.209961,0.0,163293.093,8241009.589
1,1,1055.060059,0.0,163303.156,8241015.985
2,2,1156.500000,0.0,165966.361,8241159.255
3,3,1156.500000,0.0,165969.296,8241153.903
4,4,1156.500000,0.0,165969.687,8241152.825
...,...,...,...,...,...
21459,Mancial-Produtor,1270.000000,0.0,169398.681,8242534.817
21460,C3,1259.800049,0.0,169452.571,8242153.466
21461,REL.SAM.001,1260.000000,0.0,169432.021,8242063.741
21462,C2,1259.920044,0.0,169405.556,8242107.122


In [570]:
df_node['Demanda'].sum()

300.5397862549471

In [5]:
nos = pd.read_excel('Tabelas para calibração\\Samambaia\\Nos_parciais_SAM.xlsx')
# nos = pd.read_excel('Tabelas para calibração\\Gama 2\\Nos_parciais GAM2.xlsx')
# nos = pd.read_excel('Tabelas para calibração\\Recanto_Riacho2\\Reborn\\Nos atualizados RCE RF2.xlsx')
nos

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda,Join_Count_y,...,created__4,last_edi_4,last_edi_5,TAG_VAZA_1,Zonapres_1,ZonaMano_1,Gerencia_1,Validado_1,Longitude,Latitude
0,0,1,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNOALBUQUERQUE,2025-09-13 13:22:46,NaN,VRP.SAM.013,NaN,PASS,sim,-48.143932,-15.887215
1,1,2,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNOALBUQUERQUE,2025-09-13 13:22:46,NaN,VRP.SAM.013,NaN,PASS,sim,-48.143837,-15.887158
2,2,3,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNOALBUQUERQUE,2025-09-13 13:22:46,NaN,VRP.SAM.006,NaN,PASS,sim,-48.118977,-15.886225
3,3,4,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNOALBUQUERQUE,2025-09-13 13:22:46,NaN,VRP.SAM.006,NaN,PASS,sim,-48.118950,-15.886274
4,4,5,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,...,2024-08-01 18:45:45,BRUNOALBUQUERQUE,2025-09-13 13:22:46,NaN,VRP.SAM.006,NaN,PASS,sim,-48.118947,-15.886284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21493,21493,21494,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNONEVES,2026-04-13 17:07:49,NaN,VRP.SAM.016,NaN,PASS,sim,-48.060608,-15.852921
21494,21494,21495,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNONEVES,2026-04-13 17:07:49,NaN,VRP.SAM.016,NaN,PASS,sim,-48.064996,-15.856546
21495,21495,21496,0.0,0.0,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNOALBUQUERQUE,2025-09-13 13:22:46,NaN,RAP.SAM.001,NaN,PASS,sim,-48.075656,-15.874771
21496,21496,21497,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,2024-08-01 18:45:45,BRUNOALBUQUERQUE,2025-09-13 13:22:46,NaN,RAP.SAM.001,NaN,PASS,sim,-48.075664,-15.874773


In [18]:
nos.columns

Index(['Unnamed: 0', 'FID', 'Join_Count', 'TARGET_FID', 'NODENUM', 'ELEVATION',
       'DEMAND', 'Sistema', 'Localidade', 'RAP', 'UDA', 'DMC', 'BOOSTER',
       'VRP', 'created_us', 'created_da', 'last_edite', 'last_edi_1',
       'TAG_VAZAO', 'Zonapressa', 'ZonaManobr', 'GerenciaMa', 'Validado',
       'Demanda'],
      dtype='object')

In [6]:
nos["NODENUM"] = nos["NODENUM"].astype(str)

In [7]:
df_merge = nos.merge(
    df_node,
    left_on="NODENUM",
    right_on="id",
    how="left",
    suffixes=("_nos", "_dfnode")
)


In [15]:
df_merge.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y',
       ...
       'Gerencia_1', 'Validado_1', 'Longitude', 'Latitude', 'id', 'Elevação',
       'Demanda_dfnode', 'X_COORD', 'Y_COORD', 'perda_dfnodes'],
      dtype='object', length=114)

In [16]:
df_merge

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda_nos,Join_Count_y,...,Gerencia_1,Validado_1,Longitude,Latitude,id,Elevação,Demanda_dfnode,X_COORD,Y_COORD,perda_dfnodes
0,0,1,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.143932,-15.887215,1,1055.060059,0.0,163303.156,8241015.985,0.0
1,1,2,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.143837,-15.887158,2,1156.500000,0.0,165966.361,8241159.255,0.0
2,2,3,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.118977,-15.886225,3,1156.500000,0.0,165969.296,8241153.903,0.0
3,3,4,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.118950,-15.886274,4,1156.500000,0.0,165969.687,8241152.825,0.0
4,4,5,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,...,PASS,sim,-48.118947,-15.886284,5,1052.089966,0.0,162934.864,8240890.628,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21493,21493,21494,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.060608,-15.852921,21494,1206.349976,0.0,171704.223,8244531.755,0.0
21494,21494,21495,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.064996,-15.856546,21495,1239.349976,0.0,170591.136,8242496.592,0.0
21495,21495,21496,0.0,0.0,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.075656,-15.874771,21496,1239.699951,0.0,170590.200,8242496.294,0.0
21496,21496,21497,NaN,NaN,0.0,0.0,0.020421,0.0,0.0,1,...,PASS,sim,-48.075664,-15.874773,21497,1239.699951,0.0,170585.805,8242494.780,0.0


In [8]:
df_merge["perda_dfnodes"] = df_merge["Demanda_dfnode"].fillna(0) - df_merge["Consumo"]

In [9]:
print(df_merge['Demanda_nos'].sum())
print(df_merge['Demanda_dfnode'].sum())

701.6599999999999
659.8411205877637


In [10]:
print(df_merge['perda_ajustada'].sum())
print(df_merge['perda_dfnodes'].sum())

364.3633765432098
317.01442411414985


In [577]:
df_merge['Consumo'].sum()

177.92187141203704

In [96]:
df_merge['RAP'].unique()

array(['RAP.GAM.002', 'RAP.GAM.001', ' '], dtype=object)

In [24]:
# df_merge['Pattern'].unique()

In [131]:
df_merge[df_merge['Pattern'].isna()]

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda_nos,Join_Count_y,...,DMC_1,Longitude,Latitude,id,Elevação,Demanda_dfnode,X_COORD,Y_COORD,Pattern,perda_dfnodes
0,0,0,0.0,0.167882,0.167882,1.0,0.007038,0.018944,0.186826,1,...,DMC.SSB.015,-47.771869,-15.922469,NaN,NaN,NaN,NaN,NaN,NaN,-0.167882
232,232,232,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.005,-47.776984,-15.895108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
233,233,233,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.005,-47.777115,-15.895240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
234,234,234,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.005,-47.777206,-15.895349,NaN,NaN,NaN,NaN,NaN,NaN,NaN
235,235,235,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.005,-47.777347,-15.895529,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7784,7784,7784,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.006/007,-47.773818,-15.894814,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7785,7785,7785,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.006/007,-47.773764,-15.894634,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7786,7786,7786,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.006/007,-47.773922,-15.894295,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7787,7787,7787,0.0,0.000000,NaN,NaN,NaN,NaN,0.000000,1,...,DMC.SSB.006/007,-47.774083,-15.893979,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [126]:
a = df_merge[df_merge['RAP']=='RAP.RCE.001']
a['Demanda_dfnode'].sum()

0.0

In [84]:
df_merge.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y', 'TARGET_FID', 'Join_Cou_1', 'TARGET_F_1', 'Join_Cou_2',
       'TARGET_F_2', 'ASSETGROUP', 'ASSETTYPE', 'FROMDEVICE', 'TODEVICETE',
       'GLOBALID', 'creationda', 'creator', 'lastupdate', 'updatedby',
       'installdat', 'notes', 'diameter', 'lifecycles', 'inserviced',
       'retireddat', 'material', 'designtype', 'codunidade', 'posicionam',
       'contratoob', 'tiposistem', 'tipodesenh', 'rugosidade', 'dataimplan',
       'Shape__Len', 'ORIG_FID', 'ORIG_SEQ', 'Xinicial', 'Yinicial', 'Xfinal',
       'Yfinal', 'Cota', 'POINT_X', 'POINT_Y', 'POINT_Z', 'POINT_M', 'Sistema',
       'Localidade', 'RAP', 'UDA', 'DMC', 'BOOSTER', 'VRP', 'created_us',
       'created_da', 'last_edite', 'last_edi_1', 'TAG_VAZAO', 'Zonapressa',
       'ZonaManobr', 'GerenciaMa', 'Validado', 'OBJECTID', 'NOME',
       'SHAPE_Leng', 'REGIÃO', '

In [72]:
df_merge['DMC_1'].unique()

array(['DMC.SSB.015', 'DMC.SSB.009/10', 'DMC.SSB.011', 'DMC.SSB.003/4',
       'DMC.SSB.006/007', 'DMC.SSB.001', 'DMC.SSB.012', 'DMC.SSB.005',
       'DMC.SSB.008', 'DMC.SSB.002', 'DMC.SSB.013', 'DMC.SSB.014', ' '],
      dtype=object)

In [12]:
df_merge.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y',
       ...
       'Gerencia_1', 'Validado_1', 'Longitude', 'Latitude', 'id', 'Elevação',
       'Demanda_dfnode', 'X_COORD', 'Y_COORD', 'perda_dfnodes'],
      dtype='object', length=114)

### Criando resumo de Demanda (DMC,RAP,UDA)

In [29]:
df_merge['Zonapres_1'].unique()

array(['VRP.SAM.013', 'VRP.SAM.006', 'VRP.SAM.007', 'VRP.SAM.011',
       'VRP.SAM.020', 'VRP.SAM.012', 'VRP.SAM.025', 'RAP.SAM.001',
       'VRP.SAM.024', 'VRP.SAM.005', 'VRP.SAM.008', 'VRP.SAM.019',
       'VRP.SAM.016', 'VRP.SAM.001', 'VRP.SAM.002', 'VRP.SAM.015',
       'REL.SAM.001', 'VRP.SAM.009', 'VRP.SAM.027', 'VRP.SAM.017',
       'VRP.SAM.023', 'VRP.SAM.026', 'VRP.SAM.004', 'VRP.SAM.014',
       'VRP.SAM.022', 'VRP.SAM.021', 'VRP.SAM.003', 'VRP.SAM.029',
       'VRP.SAM.030', 'VRP.SAM.028', nan, 'VRP.SAM.018', 'ERE.TAG.001'],
      dtype=object)

In [ ]:
# Lista de UDAs que você quer analisar
# rap_alvo = ['Residencial São Francisco', 'Guarapari', 'Salomão Elias',
#        'Nova Betânia I e II', 'Residencial Dom Francisco',
#        ' Residencial Buritis I e II', 'Residencial Galileia',
#        'Residencial Dom Pedro', 'Guarapari 2', 'Nova Betânia', 'R Rocio']

rap_alvo =['VRP.SAM.013', 'VRP.SAM.006', 'VRP.SAM.007', 'VRP.SAM.011',
       'VRP.SAM.020', 'VRP.SAM.012', 'VRP.SAM.025', 'RAP.SAM.001',
       'VRP.SAM.024', 'VRP.SAM.005', 'VRP.SAM.008', 'VRP.SAM.019',
       'VRP.SAM.016', 'VRP.SAM.001', 'VRP.SAM.002', 'VRP.SAM.015',
       'REL.SAM.001', 'VRP.SAM.009', 'VRP.SAM.027', 'VRP.SAM.017',
       'VRP.SAM.023', 'VRP.SAM.026', 'VRP.SAM.004', 'VRP.SAM.014',
       'VRP.SAM.022', 'VRP.SAM.021', 'VRP.SAM.003', 'VRP.SAM.029',
       'VRP.SAM.030', 'VRP.SAM.028','VRP.SAM.018']

# Filtrar apenas essas UDAs
df_filtrado = df_merge[df_merge["Zonapres_1"].isin(rap_alvo)]

# Somar os valores por UDA
# resultado = df_filtrado.groupby("cj_setor")[["Demanda_dfnode", "Consumo", "perda_dfnodes"]].sum()
resultado = df_filtrado.groupby("Zonapres_1")[["Demanda_dfnode", "Consumo","Consumo_Original", "Consumo CANF", "perda_dfnodes"]].sum()

# Mostrar no print
# for rap, row in resultado.iterrows():
#     print(f"\n{rap}")
#     print(f"  Demanda_dfnode: {row['Demanda_dfnode']:.2f}")
#     print(f"  Consumo: {row['Consumo']:.2f}")
#     print(f"  Perda_dfnodes: {row['perda_dfnodes']:.2f}")

for rap, row in resultado.iterrows():
    print(f"\n{rap}")
    if "Demanda_dfnode" in row:
        print(f"  Demanda_dfnode: {row['Demanda_dfnode']:.2f}")
    # if "Consumo" in row:
    #     print(f"  Consumo: {row['Consumo']:.2f}")
    if "Consumo_Original" in row:
        print(f"  Consumo_Original: {row['Consumo_Original']:.2f}")
    if "Consumo CANF" in row:
        print(f"  Consumo CANF: {row['Consumo CANF']:.2f}")
    if "perda_dfnodes" in row:
        print(f"  Perda_dfnodes: {row['perda_dfnodes']:.2f}")


RAP.SAM.001
  Demanda_dfnode: 164.60
  Consumo_Original: 94.33
  Consumo CANF: 0.43
  Perda_dfnodes: 68.54

REL.SAM.001
  Demanda_dfnode: 88.23
  Consumo_Original: 51.88
  Consumo CANF: 0.06
  Perda_dfnodes: 33.29

VRP.SAM.001
  Demanda_dfnode: 55.14
  Consumo_Original: 27.94
  Consumo CANF: 0.22
  Perda_dfnodes: 26.76

VRP.SAM.002
  Demanda_dfnode: 1.06
  Consumo_Original: 0.04
  Consumo CANF: 0.02
  Perda_dfnodes: 0.46

VRP.SAM.003
  Demanda_dfnode: 3.80
  Consumo_Original: 0.18
  Consumo CANF: 0.00
  Perda_dfnodes: 3.62

VRP.SAM.004
  Demanda_dfnode: 3.11
  Consumo_Original: 3.67
  Consumo CANF: 0.00
  Perda_dfnodes: -0.56

VRP.SAM.005
  Demanda_dfnode: 7.01
  Consumo_Original: 3.65
  Consumo CANF: 0.00
  Perda_dfnodes: 3.37

VRP.SAM.006
  Demanda_dfnode: 8.61
  Consumo_Original: 3.04
  Consumo CANF: 0.00
  Perda_dfnodes: 5.58

VRP.SAM.007
  Demanda_dfnode: 25.57
  Consumo_Original: 10.43
  Consumo CANF: 2.23
  Perda_dfnodes: 12.92

VRP.SAM.008
  Demanda_dfnode: 9.72
  Consumo_Orig

In [6]:
# area = pd.read_excel('Tabelas para calibração\\Recanto_Riacho2\\ligacoes_monjolo.xlsx')
# area.columns

In [7]:
# area = area[['FID']]

In [8]:
# area.value_counts().sum()

In [9]:
# nos = pd.merge(nos,area,left_on='NODENUM',right_on='FID',how='inner')
# nos

In [141]:
# nos = pd.merge(nos,area,left_on='ID',right_on='ID',how='inner')
# nos

In [ ]:
# nos_sem_match = pd.merge(
#     nos, area,
#     on="ID", 
#     how="left", 
#     indicator=True
# )

# # manter apenas os que não tiveram correspondência no 'area'
# nos_sem_match = nos_sem_match[nos_sem_match["_merge"] == "left_only"]


In [ ]:
# nos_dmc = nos_sem_match.copy()

In [ ]:
# nos_dmc = nos[~nos['VRP'].isin(['VRP.PRN.006', 'VRP.PRN.008'])]

### Tratando normal

In [16]:
df_merge.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y',
       ...
       'Gerencia_1', 'Validado_1', 'Longitude', 'Latitude', 'id', 'Elevação',
       'Demanda_dfnode', 'X_COORD', 'Y_COORD', 'perda_dfnodes'],
      dtype='object', length=114)

In [578]:
nos = df_merge.copy()

In [331]:
# nos['cj_setor'].unique()

In [ ]:
# # Substituir NaN ou None por string vazia
# nos['Pattern'] = nos['Pattern'].fillna("")

# # Substituir qualquer string que seja apenas espaços ou 'nan' (como texto) por string vazia
# nos['Pattern'] = nos['Pattern'].replace({"nan": "", "NaN": "", " ": ""})

# # Garantir que todos os valores sejam strings
# nos['Pattern'] = nos['Pattern'].astype(str)


In [29]:
nos.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y',
       ...
       'Gerencia_1', 'Validado_1', 'Longitude', 'Latitude', 'id', 'Elevação',
       'Demanda_dfnode', 'X_COORD', 'Y_COORD', 'perda_dfnodes'],
      dtype='object', length=114)

In [19]:
# nos['NOME'].unique

### Continuando 

In [20]:
# nos['DMC_1'].unique()

In [52]:
nos[nos['Zonapres_1'].isna()]

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda_nos,Join_Count_y,...,Gerencia_1,Validado_1,Longitude,Latitude,id,Elevação,Demanda_dfnode,X_COORD,Y_COORD,perda_dfnodes
9156,9156,9157,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.135443,-15.893251,9157,1093.430054,0.00000,164212.863,8240355.143,NaN
9157,9157,9158,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.135445,-15.893247,9158,1260.000000,0.00000,169379.158,8242217.470,NaN
9418,9418,9419,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.133834,-15.892517,9419,1102.060059,0.00000,164381.766,8240437.275,NaN
9419,9419,9420,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.133858,-15.892528,9420,1100.329956,0.00000,164360.188,8240426.739,NaN
9420,9420,9421,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.134061,-15.892621,9421,1099.849976,0.00000,164342.258,8240417.447,NaN
9421,9421,9422,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.134229,-15.892702,9422,1098.989990,0.00000,164324.293,8240408.434,NaN
9422,9422,9423,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.134398,-15.892781,9423,1098.170044,0.00000,164306.077,8240399.468,NaN
9423,9423,9424,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.134569,-15.892859,9424,1097.109985,0.00000,164289.315,8240391.372,NaN
9424,9424,9425,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.134727,-15.892930,9425,1096.650024,0.00000,164279.480,8240386.843,NaN
9425,9425,9426,NaN,NaN,NaN,NaN,NaN,NaN,0.00000,0,...,PASS,sim,-48.134819,-15.892970,9426,1094.599976,0.00000,164238.874,8240367.274,NaN


In [18]:
nos['Zonapres_1'].unique()

array(['VRP.SAM.013', 'VRP.SAM.006', 'VRP.SAM.007', 'VRP.SAM.011',
       'VRP.SAM.020', 'VRP.SAM.012', 'VRP.SAM.025', 'RAP.SAM.001',
       'VRP.SAM.024', 'VRP.SAM.005', 'VRP.SAM.008', 'VRP.SAM.019',
       'VRP.SAM.016', 'VRP.SAM.001', 'VRP.SAM.002', 'VRP.SAM.015',
       'REL.SAM.001', 'VRP.SAM.009', 'VRP.SAM.027', 'VRP.SAM.017',
       'VRP.SAM.023', 'VRP.SAM.026', 'VRP.SAM.004', 'VRP.SAM.014',
       'VRP.SAM.022', 'VRP.SAM.021', 'VRP.SAM.003', 'VRP.SAM.029',
       'VRP.SAM.030', 'VRP.SAM.028', nan, 'VRP.SAM.018', 'ERE.TAG.001'],
      dtype=object)

In [246]:
nos.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y', 'TARGET_FID', 'ASSETGROUP', 'ASSETTYPE', 'tiposistem',
       'designtype', 'codunidade', 'lifecycles', 'material', 'diameter',
       'Shape__Len', 'rugosidade', 'posicionam', 'tipodesenh', 'contratoob',
       'FROMDEVICE', 'TODEVICETE', 'dataimplan', 'installdat', 'inserviced',
       'retireddat', 'notes', 'creator', 'creationda', 'updatedby',
       'lastupdate', 'GLOBALID', 'ORIG_FID', 'ORIG_SEQ', 'X_INI', 'Y_INI',
       'X_FIM', 'Y_FIM', 'Sistema', 'Localidade', 'RAP', 'UDA', 'DMC',
       'BOOSTER', 'VRP', 'created_us', 'created_da', 'last_edite',
       'last_edi_1', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr', 'GerenciaMa',
       'Validado', 'COTA', 'POINT_X', 'POINT_Y', 'POINT_Z', 'POINT_M',
       'Longitude', 'Latitude', 'id', 'Elevação', 'Demanda_dfnode', 'X_COORD',
       'Y_COORD', 'perda_dfnodes'],
      dt

In [32]:
# nos['DMC_1'].unique()
nos['DMC'].unique()

array([' ', 'DMC.RF2.002', 'DMC.RF2.003', 'DMC.RF2.001', 'DMC.RCE.001',
       'DMC.RCE.002'], dtype=object)

In [498]:
lista_ids = [
14730,14731,15138,15139,15140,15141,15142,15143,15244,15245,15246,15247,15248,15249,
15520,15521,15522,15610,15611,15612,15613,15614,15615,15616,15617,15618,
15894,15895,15896,15899,15900,15901,15902,15903,15904,15905,15906,15907,15908,15909,
15910,15911,15912,15913,15914,15915,15916,15917,15918,15919,15920,15921,15922,15923,
15924,15925,15926,15927,15928,15929,15930,15931,15932,15933,15934,15935,15936,15937,
15938,15939,15940,15941,15942,15943,15947,15948,15949,15950,15951,15952,15953,15954,
15955,15956,15957,15996,15997,15998,15999,16000,16001,16002,16003,16004,16005,16006,
16007,16008,16010,16020,16021,16022,16023,16024,16025,16028,
16218,16219,
16317,16318,16319,16320,16321,16322,16323,16324,16325,16336,16337,16338,
16351,16352,16355,16356,16359,16360,16361,16362,16363,16364,16368,16369,16370,16371,
16372,16373,16374,16375,16376,16377,16378,16399,16400,16416,16417,
16429,16430,16431,16432,16433,16434,16435,16436,16437,16438,16439,16440,16441,16442,
16443,16444,16445,16446,16447,16448,16449,16450,16451,
16560,16561,16562,16563,16587,16588,16589,16590,16591,16592,
16732,16733,17129,17185,17186,17245,17246,
17308,17309,17310,17315,17316,17317,17318,17319,17320,17321,
19090,19092,19097,19098,19147,19148,19149,19150,19151,19152,19153,19154,19155,19156,19157,19158,19159,
19161,19162,19163,19164,19165,19168,19169,
19770,19772,19773,20317,20342,20669,20724,20725,20727,20729,20730,20756,
20896,21230,21231,21232,21233,21234,21235,21236,21237,21238,21239,21240,21241,21242,21243,
21244,21245,21246,21247,21257,21258,21259,21260,21261,21262,21263,21264
]

In [486]:
#VRP.SAM.004

lista_ids = [
    751, 752, 987, 988, 997, 998, 999, 1000, 1010, 1011,
    1042, 1043, 1225, 1226, 2788, 2789, 2790, 2791, 2795,
    2796, 2797, 2798, 2799, 2800, 2806, 3108, 3109, 3110,
    3111, 3112, 3113, 3641, 3642, 3868, 3869, 3872, 4508,
    4509, 4512, 4513, 5052, 5053, 7503, 7505, 7506, 7507,
    7508, 7509, 7510, 7511, 7512, 7513, 7514, 7515, 7516,
    7517, 7548, 7549, 7550, 7551, 7632, 7633, 7856, 7857,
    7858, 7859, 7860, 7861, 7862, 7863, 7864, 7865, 7866,
    7867, 7877, 7878, 10694, 10695, 10696, 10697, 10698,
    10699, 10700, 10701, 10702, 10703, 10704, 12477, 12478,
    12479, 12480, 12481, 12482, 12483, 12484, 12485, 13192,
    13193,
    14730, 14731, 15138, 15139, 15140, 15141, 15142, 15143,
    15244, 15245, 15246, 15247, 15248, 15249, 15520, 15521,
    15522, 15610, 15611, 15612, 15613, 15614, 15615, 15616,
    15617, 15618
]

In [77]:
nos.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y',
       ...
       'Gerencia_1', 'Validado_1', 'Longitude', 'Latitude', 'id', 'Elevação',
       'Demanda_dfnode', 'X_COORD', 'Y_COORD', 'perda_dfnodes'],
      dtype='object', length=114)

In [554]:
nos['Zonapressa'].unique()

array(['VRP.GAM.003', 'VRP.GAM.018', 'RAP.GAM.001', 'VRP.GAM.029',
       'VRP.GAM.015', 'VRP.GAM.017', 'VRP.GAM.016', 'RAP.GAM.002',
       'VRP.GAM.028', 'VRP.GAM.007', 'VRP.GAM.013', 'VRP.GAM.027',
       'VRP.GAM.014', 'VRP.GAM.001', 'VRP.GAM.019', 'VRP.GAM.006',
       'VRP.GAM.022', 'VRP.GAM.025', 'VRP.GAM.020', 'VRP.GAM.010',
       'VRP.GAM.008', 'VRP.GAM.024', ' ', 'VRP.GAM.026', 'VRP.GAM.012',
       'VRP.GAM.021', 'VRP.GAM.011', 'VRP.GAM.023', 'VRP.GAM.004',
       'VRP.GAM.005'], dtype=object)

In [592]:
# nos_dmc = nos
# nos_dmc = nos[nos['Pattern']=='VZ3.RAP.SSB.002']
# nos_dmc = nos[nos['DMC']=='DMC.SSB.001']
# nos_dmc = nos[nos['NODENUM'].astype(int).isin(lista_ids)]
# nos_dmc = nos[nos['Zonapressa']=='VRP.SAM.001']

# nos_dmc = nos[nos['codunidade']=='SAT.NBN.014']
# nos_dmc = nos[nos['RA']=='CANDANGOLÂNDIA']
# nos_dmc = nos[nos['UDA_1']=='UDA.SAM.002']
# nos_dmc = nos[nos['Zonapres_1']=='VRP.SAM.004']
# nos_dmc = nos[nos['DMC_1']=='DMC.SAM.004']
# nos_dmc = nos[nos['RAP_1']=='REL.SAM.001']
# nos_dmc = nos[nos['DMC']=='DMC.GAM.005']
# nos_dmc = nos[nos['Zonapressa']=='VRP.GAM.023']
nos_dmc = nos[nos['TAG_VAZAO']=='VZ1.AAT.GAM.090']
# nos_dmc = nos[nos['Zonapressa'].isin(['ITP - DMC 4', 'ITP - DMC 1', 'ITP - DMC 3','ITP - DMC 2'])]
# nos_dmc = nos[nos['VRP'].isin(['VRP.PRN.006', 'VRP.PRN.008'])]

# nos_dmc = nos
nos_dmc

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda_nos,Join_Count_y,...,POINT_Z,POINT_M,Longitude,Latitude,id,Elevação,Demanda_dfnode,X_COORD,Y_COORD,perda_dfnodes
480,498,498,0.000000,0.0,0.000000,0.0,0.024679,0.0,0.000000,2,...,0.0,NaN,-48.054411,-16.023469,498,1133.709961,0.000000,173109.748,8226061.875,0.000000
481,499,499,0.000000,0.0,0.000000,0.0,0.024679,0.0,0.000000,2,...,0.0,NaN,-48.054416,-16.023497,499,1133.449951,0.000000,173109.257,8226058.752,0.000000
671,692,692,0.000000,0.0,0.000000,0.0,0.024679,0.0,0.000000,1,...,0.0,NaN,-48.053868,-16.032421,692,1096.099976,0.000000,173182.609,8225071.319,0.000000
672,693,693,0.008102,0.0,0.008102,0.0,0.024679,0.0,0.008102,1,...,0.0,NaN,-48.053796,-16.032463,693,1095.750000,0.007431,173190.359,8225066.745,-0.000671
748,775,775,0.012860,0.0,0.012860,0.0,0.024679,0.0,0.012860,1,...,0.0,NaN,-48.058694,-16.020766,775,1142.280029,0.010099,172646.618,8226354.445,-0.002761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8130,8588,8588,0.001929,0.0,0.001929,0.0,0.024679,0.0,0.001929,1,...,0.0,NaN,-48.052945,-16.028429,8588,1112.359985,0.001770,173274.896,8225514.876,-0.000159
8131,8589,8589,0.000000,0.0,0.000000,0.0,0.024679,0.0,0.000000,1,...,0.0,NaN,-48.052970,-16.028413,NaN,NaN,NaN,NaN,NaN,0.000000
8132,8590,8590,0.000000,0.0,0.000000,0.0,0.024679,0.0,0.000000,1,...,0.0,NaN,-48.052962,-16.028419,8590,1112.359985,0.000000,173273.095,8225515.978,0.000000
8133,8591,8591,0.000000,0.0,0.000000,0.0,0.024679,0.0,0.000000,1,...,0.0,NaN,-48.052956,-16.028422,8591,1112.359985,0.000000,173273.702,8225515.607,0.000000


In [593]:
nos_dmc['Zonapressa'].unique()

array(['RAP.GAM.002', 'VRP.GAM.010', 'VRP.GAM.026'], dtype=object)

In [439]:
nos_dmc = nos_dmc[nos_dmc['Zonapressa'] != 'VRP.GAM.014']

In [267]:
nos_dmc = nos[
    (nos['DMC'] == 'DMC.SSB.003/4') &
    (~nos['NODENUM'].astype(str).isin(lista_ids))
]


In [440]:
nos_dmc

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda_nos,Join_Count_y,...,POINT_Z,POINT_M,Longitude,Latitude,id,Elevação,Demanda_dfnode,X_COORD,Y_COORD,perda_dfnodes
75,79,79,0.00000,0.0,0.00000,0.0,0.024679,0.0,0.00000,2,...,0.0,NaN,-48.052861,-15.994196,79,1196.310059,0.000000,173228.115,8229306.242,0.000000
130,134,134,0.00000,0.0,0.00000,0.0,0.024679,0.0,0.00000,1,...,0.0,NaN,-48.060101,-15.993478,134,1182.310059,0.000000,172451.423,8229374.308,0.000000
131,135,135,0.02572,0.0,0.02572,0.0,0.024679,0.0,0.02572,1,...,0.0,NaN,-48.059921,-15.994048,135,1182.119995,0.021627,172471.589,8229311.540,-0.004093
132,136,136,0.00463,0.0,0.00463,0.0,0.024679,0.0,0.00463,1,...,0.0,NaN,-48.059658,-15.994901,136,1181.680054,0.003893,172501.132,8229217.502,-0.000737
262,273,273,0.00000,0.0,0.00000,0.0,0.024679,0.0,0.00000,1,...,0.0,NaN,-48.059353,-15.996901,273,1181.160034,0.000000,172537.142,8228996.468,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7721,8141,8141,0.00000,0.0,0.00000,0.0,0.024679,0.0,0.00000,1,...,0.0,NaN,-48.059605,-15.997545,8141,1179.650024,0.000000,172511.111,8228924.745,0.000000
7722,8142,8142,0.00000,0.0,0.00000,0.0,0.024679,0.0,0.00000,1,...,0.0,NaN,-48.052889,-15.994089,8142,1196.689941,0.000000,173224.996,8229318.042,0.000000
7723,8143,8143,0.00000,0.0,0.00000,0.0,0.024679,0.0,0.00000,1,...,0.0,NaN,-48.052896,-15.994089,8143,1196.689941,0.000000,173224.214,8229318.004,0.000000
7738,8173,8173,0.00000,0.0,0.00000,0.0,0.024679,0.0,0.00000,1,...,0.0,NaN,-48.052925,-15.994101,8173,1196.689941,0.000000,173221.138,8229316.616,0.000000


In [ ]:
# vrps_alvo = [
#     'VRP.RCE.001', 
#     'VRP.RCE.016', 
#     'VRP.RCE.013', 
#     'VRP.RCE.012', 
#     'VRP.RCE.011', 
#     'VRP.RCE.014', 
#     'VRP.REC.015'
# ]

# nos_dmc = nos[nos['VRP'].isin(vrps_alvo)]

In [465]:
nos_dmc.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo_Original', 'Consumo CANF', 'Consumo',
       'Join_Count_x', 'perda', 'perda_ajustada', 'Demanda_nos',
       'Join_Count_y',
       ...
       'Gerencia_1', 'Validado_1', 'Longitude', 'Latitude', 'id', 'Elevação',
       'Demanda_dfnode', 'X_COORD', 'Y_COORD', 'perda_dfnodes'],
      dtype='object', length=114)

In [33]:
nos_dmc= nos.copy()

In [ ]:
# nos_dmc['VRP'].unique()

array([' '], dtype=object)

In [583]:
nos_dmc['Consumo'].sum()

4.300218364197531

In [468]:
nos_dmc['Consumo CANF'].sum()

0.0

In [584]:
nos_dmc['perda_dfnodes'].sum()

0.3872175865662257

In [586]:
nos_dmc['Demanda_dfnode'].sum()

4.687435950763756

In [170]:
# nos_vrp[nos_vrp.isna()]

In [171]:
# vrpcalibra = nos[nos['NODENUM'].isin(nos_vrp)]
# vrpcalibra 

In [172]:
# filtro = ligacao[ligacao['Input_FID_y']==10848]
# filtro

In [174]:
# ligacao['Perda'] = 170.02358/len(ligacao)

In [173]:
# ligacaoxperda = ligacao.groupby(by='Input_FID_y')['Perda'].sum().reset_index()
# ligacaoxperda

In [175]:
# lista_nos = pd.merge(uniforme,ligacaoxperda,left_on='NODENUM_x',right_on='Input_FID_y',how='left')

In [176]:
# lista_nos = lista_nos[['NODENUM_x','Consumo','Perda']]
# lista_nos['Consumo'].fillna(0,inplace=True)
# lista_nos['Perda'].fillna(0,inplace=True)

In [177]:
# lista_nos['Consumo'].fillna(0,inplace=True)

In [178]:
# lista_nos['Demanda']= lista_nos['Consumo']+lista_nos['Perda']

In [179]:
# d.getnodedema
# dict_ids['18']

In [180]:
# list_NODENUM == 2940


In [181]:
# vrpcalibra

In [ ]:
# nos_dmc=nos.copy()

In [276]:
nos_dmc

,Unnamed: 0,NODENUM,Consumo_Original,Consumo CANF,Consumo,Join_Count_x,perda,perda_ajustada,Demanda_nos,Join_Count_y,...,POINT_Z,POINT_M,Longitude,Latitude,id,Elevação,Demanda_dfnode,X_COORD,Y_COORD,perda_dfnodes
33,33,33,0.000000,0.0,0.000000,0.0,0.024679,0.000000,0.000000,1,...,0.000,NaN,-48.050729,-16.011217,33,1167.000000,0.000000,173484.258,8227424.567,0.000000e+00
34,34,34,0.000000,0.0,0.000000,0.0,0.024679,0.000000,0.000000,1,...,0.000,NaN,-48.051215,-16.010915,34,1168.640015,0.000000,173431.708,8227457.257,0.000000e+00
39,39,39,0.000000,0.0,0.000000,0.0,0.024679,0.000000,0.000000,1,...,0.000,NaN,-48.052458,-16.013848,39,1163.609985,0.000000,173303.268,8227130.436,0.000000e+00
40,40,40,0.001543,0.0,0.001543,0.0,0.024679,0.000000,0.001543,1,...,0.000,NaN,-48.051916,-16.014192,40,1161.390015,0.001543,173362.653,8227094.119,-2.098853e-07
46,46,46,0.000000,0.0,0.000000,1.0,0.024679,0.802961,0.802961,2,...,0.000,NaN,-48.051736,-16.010580,46,1170.510010,0.802961,173375.308,8227493.557,8.029610e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7605,8004,8004,0.000000,0.0,0.000000,0.0,0.024679,0.000000,0.000000,1,...,1139.320,NaN,-48.047280,-16.013345,8004,1139.930054,0.000000,173857.098,8227194.325,0.000000e+00
7606,8005,8005,0.000000,0.0,0.000000,0.0,0.024679,0.000000,0.000000,1,...,1140.385,NaN,-48.047399,-16.013547,8005,1139.339966,0.000000,173844.660,8227171.746,0.000000e+00
7607,8006,8006,0.000000,0.0,0.000000,0.0,0.024679,0.000000,0.000000,1,...,0.000,NaN,-48.047437,-16.013611,8006,1139.270020,0.000000,173840.691,8227164.655,0.000000e+00
8057,8496,8496,0.000000,0.0,0.000000,0.0,0.024679,0.000000,0.000000,1,...,0.000,NaN,-48.047280,-16.013722,NaN,NaN,NaN,NaN,NaN,0.000000e+00


In [74]:
nos_dmc.columns

Index(['Unnamed: 0', 'NODENUM', 'Consumo', 'Join_Count_x', 'perda',
       'perda_ajustada', 'Demanda_nos', 'Join_Count_y', 'TARGET_FID',
       'Join_Cou_1', 'TARGET_F_1', 'ASSETGROUP', 'ASSETTYPE', 'tiposistem',
       'designtype', 'codunidade', 'lifecycles', 'material', 'diameter',
       'Shape__Len', 'rugosidade', 'posicionam', 'tipodesenh', 'contratoob',
       'FROMDEVICE', 'TODEVICETE', 'dataimplan', 'installdat', 'inserviced',
       'retireddat', 'notes', 'creator', 'creationda', 'updatedby',
       'lastupdate', 'GLOBALID', 'ORIG_FID', 'ORIG_SEQ', 'X_INI', 'Y_INI',
       'X_FIM', 'Y_FIM', 'Sistema', 'Localidade', 'RAP', 'UDA', 'DMC',
       'BOOSTER', 'VRP', 'created_us', 'created_da', 'last_edite',
       'last_edi_1', 'TAG_VAZAO', 'Zonapressa', 'ZonaManobr', 'GerenciaMa',
       'Validado', 'COTA', 'POINT_X', 'POINT_Y', 'POINT_Z', 'POINT_M',
       'OBJECTID', 'NOME', 'SHAPE_Leng', 'REGIÃO', 'RA', 'Shape_Le_1',
       'Shape_Area', 'Longitude', 'Latitude', 'id', 'Elevaçã

In [587]:
# lista_nos = lista_nos[['NODENUM_x','Demanda']]

# lista_nos = nos_dmc [['NODENUM','Demanda_nos']]
lista_nos = nos_dmc [['NODENUM','Demanda_dfnode']]
# lista_nos = nos_dmc [['ID','Demand']]
# lista_nos = nos [['NODENUM','Demanda']]

In [588]:
lista_nos['Demanda_dfnode'].sum()
# lista_nos['Demanda_nos'].sum()

4.687435950763756

In [589]:
# lista_nos = nos.copy()
lista_nos = lista_nos[['NODENUM','Demanda_dfnode']]
# lista_nos = lista_nos[['NODENUM','Demanda_nos']]
# lista_nos = lista_nos[['NODENUM','Demanda']]
lista_nos = lista_nos.dropna(subset=['NODENUM'])
lista_nos

,NODENUM,Demanda_dfnode
1285,1321,0.000000
1286,1322,0.017063
1287,1323,0.026743
1288,1324,0.002689
1289,1325,0.000000
...,...,...
7173,7510,0.000000
7174,7511,0.002127
7252,7589,0.000000
7253,7590,0.000000


In [74]:
# lista_nos ['NODENUM'] = lista_nos['NODENUM'].astype(int)

In [590]:
#Criando uma lista com o ID dos Nós que serão calibrados

list_NODENUM = lista_nos['NODENUM'].values.tolist()
list_Demanda_final = lista_nos['Demanda_dfnode'].values.tolist()
# list_Demanda_final = lista_nos['Demanda_nos'].values.tolist()

In [20]:
# import pandas as pd

# # valor total a distribuir
# total = 11.28

# # quantidade de linhas
# n = len(nos_dmc)

# # valor por linha
# valor_por_linha = total / n  

# # atribui igualmente
# nos_dmc["Demanda"] = valor_por_linha  

# # garante que a soma é igual ao total
# print(nos_dmc)
# print("Soma final:", nos_dmc["Demanda"].sum())


In [ ]:
# lista_nos = nos_dmc [['NODENUM','Demand']]

In [ ]:
# lista_nos['Demand'].sum()

11.280000000000005

In [21]:
# #Criando uma lista com o ID dos Nós que serão calibrados
# # lista_nos = nos.copy()
# lista_nos = lista_nos[['NODENUM','Demand']]
# lista_nos = lista_nos.dropna(subset=['NODENUM'])
# lista_nos

# list_NODENUM = lista_nos['NODENUM'].values.tolist()
# list_Demanda_final = lista_nos['Demand'].values.tolist()

In [23]:
# Garante que os IDs estão no formato correto (str para comparação)
dict_novas_demandas = {
    str(int(id)): round(float(demanda), 4)
    for id, demanda in zip(list_NODENUM, list_Demanda_final)
}
dict_novas_demandas

{'0': 7.36,
 '1': 0.0,
 '2': 0.0049,
 '3': 0.0177,
 '4': 0.0,
 '5': 0.0,
 '6': 0.0612,
 '7': 0.0,
 '8': 0.0028,
 '9': 0.0148,
 '10': 0.0,
 '11': 0.0422,
 '12': 0.0,
 '13': 0.0,
 '14': 0.0087,
 '15': 0.0,
 '16': 0.0,
 '17': 0.0031,
 '18': 0.0,
 '19': 0.0167,
 '20': 0.0334,
 '21': 0.0305,
 '22': 0.0233,
 '23': 0.0,
 '24': 0.0,
 '25': 0.0246,
 '26': 0.0,
 '27': 0.0279,
 '28': 0.021,
 '29': 0.0365,
 '30': 0.0198,
 '31': 0.0,
 '32': 0.0,
 '33': 0.0,
 '34': 0.0,
 '35': 0.0087,
 '36': 0.0962,
 '37': 0.055,
 '38': 0.0628,
 '39': 0.0,
 '40': 0.0015,
 '41': 0.0385,
 '42': 0.0267,
 '43': 0.0,
 '44': 0.0067,
 '45': 0.0055,
 '46': 0.803,
 '47': 0.0,
 '48': 0.0,
 '49': 0.0,
 '50': 0.0,
 '51': 0.0,
 '52': 0.0,
 '53': 0.0,
 '54': 0.0,
 '55': 0.0019,
 '56': 0.0231,
 '57': 0.0,
 '58': 0.0004,
 '59': 0.0,
 '60': 0.0091,
 '61': 0.0557,
 '62': 0.8414,
 '63': 0.0736,
 '64': 0.0,
 '65': 0.0691,
 '66': 0.0,
 '67': 0.0,
 '68': 0.0,
 '69': 0.0,
 '70': 0.0,
 '71': 0.0,
 '76': 0.0,
 '77': 0.0,
 '78': 0.0,
 '79': 

In [45]:
dict_novas_demandas = {str(id): float(demanda) for id, demanda in zip(list_NODENUM, list_Demanda_final)}


In [50]:
qtd_nos_mapeados = len(dict_novas_demandas)
qtd_nos_mapeados


2888

In [402]:
d.getLinkStatus()

array([0, 0, 0, ..., 0, 0, 0])

# Atribuindo padrão de consumo ao nós

In [263]:
d.getNodeDemandPatternNameID()

{1: ['VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.010',
  'VZ1.AAT.RCE.010',
  'Teste',
  'Teste',
  'Teste',
  'Teste',
  'Teste',
  'Teste',
  'Teste',
  'Teste',
  'Teste',
  '',
  '',
  'VZ1.AAT.RCE.010',
  'VZ1.AAT.RCE.010',
  '',
  '',
  'VZ1.AAT.RCE.010',
  'VZ1.AAT.RCE.010',
  'VZ1.AAT.RCE.010',
  'VZ1.AAT.RCE.010',
  'Teste',
  'Teste',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.AAT.RCE.030',
  'VZ1.DMC.RCE.003',
  'VZ1.DMC.RCE.003',
  'VZ1.AAT.R

In [ ]:
d.setOptionsPatternDemandMultiplier()

In [287]:
# ==============================================================================
# 1. CRIAÇÃO DO PADRÃO (Com nome sem espaços)
# ==============================================================================

patternID = 'PadraoVRP.RCE.002' # <<-- NOME CORRIGIDO SEM ESPAÇO

# FATORES MULTIPLICADORES: (24 fatores)
patternMult = np.array([
    0.815, 0.677, 0.568, 0.517, 0.495, 
    0.514, 0.601, 0.743, 0.848, 0.940, 
    1.075, 1.186, 1.361, 1.404, 1.300, 
    1.215, 1.173, 1.179, 1.214, 1.307, 
    1.308, 1.247, 1.160, 0.983
])

# Tenta remover o padrão, mesmo que não apareça na lista, para limpar a memória.
try:
    # Obtém o índice, se ele "existe" na memória
    patternIndex = d.getPatternIndex(patternID)
    print(f"Padrão '{patternID}' encontrado (ID: {patternIndex}). TENTANDO REMOVER...")
    
    # Remove o padrão "fantasma"
    d.deletePattern(patternIndex)
    print(f"Padrão '{patternID}' removido com sucesso.")

except Exception:
    # Se não encontrar ou não conseguir remover, ele será criado em seguida
    print(f"Padrão '{patternID}' não encontrado ou impossível de remover, será criado.")

# 2. CRIAÇÃO GARANTIDA
print(f"Recriando e definindo o padrão '{patternID}'...")
try:
    patternIndex = d.addPattern(patternID)
    d.setPattern(patternIndex, patternMult)
    print(f"Padrão '{patternID}' criado com sucesso e índice: {patternIndex}.")
except Exception as e:
    print(f"ERRO CRÍTICO na criação do padrão! Verifique a lista patternMult. Detalhe: {e}")
    # Se falhar aqui, o restante do código não funcionará
    exit()

# ==============================================================================
# 3. ATRIBUIÇÃO E RASTREAMENTO
# ==============================================================================

# Inicialização
dict_ids = {nome: index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
nos_atribuidos_sucesso = []
contador_sucesso = 0
INDICE_DEMANDA_BASE = 1 # Índice 1 para a primeira (base) demanda do nó.

print("\nIniciando a atribuição de padrões...")

for id_no in list_NODENUM:
    nome_no = str(id_no) 
    
    if nome_no in dict_ids:
        try:
            nodeIndex = dict_ids[nome_no]
            
            # ATRIBUIÇÃO CORRIGIDA: Usa 3 argumentos (nodeIndex, demandIndex=1, patternIndex)
            d.setNodeDemandPatternIndex(nodeIndex, INDICE_DEMANDA_BASE, patternIndex) 
            
            nos_atribuidos_sucesso.append(nome_no)
            contador_sucesso += 1
            
        except Exception as e:
            print(f"ERRO DE ATRIBUIÇÃO: Falha ao atribuir o padrão ao nó '{nome_no}'. Detalhe: {e}")
            
    else:
        print(f"AVISO: O nó '{nome_no}' não foi encontrado no modelo e foi ignorado.")

# ==============================================================================
# 4. RESULTADOS
# ==============================================================================

print("\n" + "="*70)
print(f"RESUMO DA ATRIBUIÇÃO DO PADRÃO '{patternID}'")
print("="*70)
print(f"QUANTIDADE DE NÓS ATRIBUÍDOS COM SUCESSO: {contador_sucesso}")

if contador_sucesso > 0:
    print("LISTA DE NÓS ATRIBUÍDOS COM SUCESSO:")
    # Divide a lista em blocos de 10 para melhor visualização
    blocos_nos = [nos_atribuidos_sucesso[i:i + 10] for i in range(0, len(nos_atribuidos_sucesso), 10)]
    for bloco in blocos_nos:
        print(f"  {', '.join(bloco)}")
else:
    print("NENHUM NÓ DA LISTA RECEBEU O PADRÃO COM SUCESSO.")
print("="*70)

Padrão 'PadraoVRP.RCE.002' encontrado (ID: 11). TENTANDO REMOVER...
Padrão 'PadraoVRP.RCE.002' removido com sucesso.
Recriando e definindo o padrão 'PadraoVRP.RCE.002'...
Padrão 'PadraoVRP.RCE.002' criado com sucesso e índice: 11.

Iniciando a atribuição de padrões...
AVISO: O nó '11785' não foi encontrado no modelo e foi ignorado.
AVISO: O nó '11970' não foi encontrado no modelo e foi ignorado.
AVISO: O nó '12007' não foi encontrado no modelo e foi ignorado.
AVISO: O nó '12069' não foi encontrado no modelo e foi ignorado.
AVISO: O nó '13438' não foi encontrado no modelo e foi ignorado.

RESUMO DA ATRIBUIÇÃO DO PADRÃO 'PadraoVRP.RCE.002'
QUANTIDADE DE NÓS ATRIBUÍDOS COM SUCESSO: 925
LISTA DE NÓS ATRIBUÍDOS COM SUCESSO:
  27, 28, 31, 32, 335, 336, 337, 338, 339, 340
  341, 342, 343, 344, 345, 346, 347, 348, 349, 350
  353, 354, 376, 377, 378, 379, 380, 381, 382, 383
  384, 385, 386, 387, 388, 389, 390, 391, 392, 393
  394, 395, 396, 668, 669, 676, 677, 747, 748, 822
  823, 824, 825, 826

In [271]:
d.getPatternIndex()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [283]:
d.getPatternNameID()

['1',
 'VZ1.AAT.RCE.010',
 'VZ1.AAT.RCE.030',
 'VZ1.DMC.RCE.001',
 'VZ1.DMC.RCE.002',
 'VZ1.EBO.GAM.001',
 'VZ1.DMC.RCE.003',
 'EBO.GAM.001',
 'ERE.RCE.001',
 'Teste',
 'PadraoVRP.RCE.002']

In [288]:
d.saveInputFile('Epanet Gerados\\Recanto_Riacho2\\Calibracao\\RCE_RF2_V14_teste.inp')

# Tentativa de fazer o que o Watergems faz

In [1091]:
# from epyt import epanet2 as d
from scipy.optimize import minimize
import numpy as np
import datetime

ids_nós = {nome:index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}
ids_nós

# Lista de nós a ajustar (dict: ID -> valor inicial)
dict_novas_demandas = {str(int(id)): float(demanda) for id, demanda in zip(list_NODENUM, list_Demanda_final)}
ids_para_ajuste = list(dict_novas_demandas.keys())

# Pressão observada (24 valores, um para cada hora)
pressao_observada = np.array([90.65, 91.22, 91.42, 91.51, 91.45, 91.15, 89.44, 88.49, 
                              87.61, 86.95, 86.33, 85.69, 85.87, 86.24, 86.55, 86.68, 
                              86.67, 86.59, 86.13, 86.26, 86.74, 87.23, 88.87, 89.85])  # <- insira seus 24 valores aqui



# Função que define o erro total entre a pressão simulada e observada
def erro_total(demandas):
    for node_id, nova_demanda in zip(ids_para_ajuste, demandas):
        d.setNodeBaseDemands(node_id, nova_demanda)

    # Rodar a simulação hidráulica
    d.openHydraulicAnalysis()
    d.initializeHydraulicAnalysis()
    
    pressao_simulada = []
    tstep = 1
    while tstep > 0:
        d.runHydraulicAnalysis()
        pressao = d.getNodePressure()
        pressao_simulada.append(pressao[ids_nós["PM.VRP.VCP.013"]])
        tstep = d.nextHydraulicAnalysisStep()
    
    d.closeHydraulicAnalysis()

    # Cortar para o menor comprimento entre simulado e observado
    min_len = min(len(pressao_simulada), len(pressao_observada))
    pressao_simulada = np.array(pressao_simulada[:min_len])
    pressao_obs = np.array(pressao_observada[:min_len])

    # Calcular erro quadrático total
    erro = np.sum((pressao_simulada - pressao_obs)**2)
    return erro



# Valores iniciais
valores_iniciais = list(dict_novas_demandas.values())

# Garantir que nenhum valor inicial seja zero ou negativo
valores_iniciais = [max(0.01, v) for v in valores_iniciais]

# Agora os limites são construídos com a certeza de que a demanda inicial é positiva
bounds = []
for demanda in valores_iniciais:
    inferior = max(0.01, demanda * 0.1)
    superior = max(inferior + 0.001, demanda * 5)  # Garante que superior > inferior
    bounds.append((inferior, superior))


# Otimização
resultado = minimize(
    erro_total,
    valores_iniciais,
    method='L-BFGS-B',
    bounds = [(max(0.01, demanda * 0.1), demanda * 5) for demanda in valores_iniciais],  # Limites razoáveis
    options={'disp': True}
)

# Atualiza as demandas no modelo com as otimizadas
if resultado.success:
    demandas_otimizadas = resultado.x
    for node_id, nova_demanda in zip(ids_para_ajuste, demandas_otimizadas):
        d.setNodeBaseDemand(node_id, nova_demanda)
    print("Calibração finalizada com sucesso.")
else:
    print("Falha na otimização:", resultado.message)


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 203: function call contains undefined node
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 203: function call contains undefined node
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 203: function call contains undefined node
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 203: function call contains undefined node
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 203: function call contains undefined node
  warnings.warn(errmssg.value.decode())
c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packa

KeyboardInterrupt: 

In [597]:
dict_name_index['0']

1

In [600]:
d.getNodeBaseDemands(dict_name_index['10848'])

{1: array([0.72089893])}

# Substituindo demanda antiga por nova

In [ ]:
dict_novas_demandas = {str(id): float(demanda) for id, demanda in zip(list_NODENUM, list_Demanda_final)}

In [46]:
# Calcular a demanda total da rede com base no formato correto retornado
demanda_total = 0

for i in range(d.getNodeCount()):
    demandas = d.getNodeBaseDemands(i)

    if isinstance(demandas, dict):
        for val in demandas.values():
            if isinstance(val, (list, tuple, np.ndarray)):
                demanda_total += sum(val)
            elif isinstance(val, (int, float)):
                demanda_total += val
    else:
        print(f"Aviso: formato inesperado na demanda do nó {i} → {demandas}")

print("📊 Demanda total da rede:", demanda_total)

📊 Demanda total da rede: 420.1737064467161


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 203: function call contains undefined node
  warnings.warn(errmssg.value.decode())


In [47]:
# -------------------------------------------------------------
# MAPEAMENTO NOME → ÍNDICE DOS NÓS EXISTENTES NO MODELO
# -------------------------------------------------------------
dict_name_index = {
    str(nome).strip(): index
    for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())
}

# -------------------------------------------------------------
# CONTADORES E LISTAS DE CONTROLE
# -------------------------------------------------------------
nao_encontrados = []
qtd_editados = 0
qtd_iguais = 0

nos_editados = []
nos_iguais = []

# -------------------------------------------------------------
# LOOP DE APLICAÇÃO DAS NOVAS DEMANDAS
# -------------------------------------------------------------
for id_nome, new_demand in dict_novas_demandas.items():
    nome_limpo = str(id_nome).strip()

    if nome_limpo in dict_name_index:
        try:
            id_index = dict_name_index[nome_limpo]

            # Demanda atual
            demanda_atual = d.getNodeBaseDemands(id_index)
            antiga = list(demanda_atual.values())[0][0] if demanda_atual else 0.0

            # Só altera se for diferente (com tolerância)
            if abs(antiga - new_demand) > 1e-6:
                d.setNodeBaseDemands(id_index, new_demand)
                qtd_editados += 1
                nos_editados.append(nome_limpo)

                print(
                    f"✏️ Nó {nome_limpo:<12} "
                    f"(Índice {id_index:<4}) | "
                    f"{antiga:.3f} → {new_demand:.3f}"
                )
            else:
                qtd_iguais += 1
                nos_iguais.append(nome_limpo)

                print(
                    f"⏭️ Nó {nome_limpo:<12} "
                    f"(Índice {id_index:<4}) | "
                    f"Sem alteração ({antiga:.3f})"
                )

        except Exception as e:
            print(f"⚠️ Erro ao ajustar {nome_limpo}: {e}")

    else:
        nao_encontrados.append((nome_limpo, new_demand))

# -------------------------------------------------------------
# RELATÓRIO FINAL
# -------------------------------------------------------------
print("\n" + "—" * 70)

# Não encontrados
if nao_encontrados:
    print("❌ Nós não encontrados no modelo:")
    soma_nao_encontrados = 0.0
    nao_nulos = []

    for nome, valor in nao_encontrados:
        print(f" - {nome:<15} | Demanda prevista: {valor:.3f}")
        if valor != 0:
            nao_nulos.append((nome, valor))
            soma_nao_encontrados += valor

    if nao_nulos:
        print("\n🔎 Nós não encontrados com demanda > 0:")
        for nome, valor in nao_nulos:
            print(f" - {nome:<15} | Demanda: {valor:.3f}")

    print("\n📊 Resumo das demandas não encontradas:")
    print(f"   Total de nós não encontrados: {len(nao_encontrados)}")
    print(f"   Soma das demandas não nulas: {soma_nao_encontrados:.3f}")
else:
    print("✅ Todos os nós foram encontrados no modelo.")

# -------------------------------------------------------------
# RESUMO GERAL DE EDIÇÕES
# -------------------------------------------------------------
print("\n📊 Resumo geral do processamento:")
print(f"   ✏️ Nós com demanda editada: {qtd_editados}")
print(f"   ⏭️ Nós sem alteração:       {qtd_iguais}")
print(f"   ❌ Nós não encontrados:     {len(nao_encontrados)}")
print(f"   📌 Total processados:       {qtd_editados + qtd_iguais + len(nao_encontrados)}")

print("—" * 70)


✏️ Nó 0            (Índice 1   ) | 0.000 → 5.550
✏️ Nó 1            (Índice 2   ) | 0.002 → 0.001
⏭️ Nó 2            (Índice 3   ) | Sem alteração (0.000)
⏭️ Nó 3            (Índice 4   ) | Sem alteração (0.000)
⏭️ Nó 4            (Índice 5   ) | Sem alteração (0.000)
✏️ Nó 5            (Índice 6   ) | 0.011 → 0.018
⏭️ Nó 6            (Índice 7   ) | Sem alteração (0.000)
⏭️ Nó 7            (Índice 8   ) | Sem alteração (0.000)
⏭️ Nó 8            (Índice 9   ) | Sem alteração (0.000)
✏️ Nó 9            (Índice 10  ) | 0.022 → 0.027
✏️ Nó 10           (Índice 11  ) | 0.023 → 0.022
⏭️ Nó 11           (Índice 12  ) | Sem alteração (0.000)
⏭️ Nó 12           (Índice 13  ) | Sem alteração (0.000)
✏️ Nó 13           (Índice 14  ) | 0.118 → 0.055
⏭️ Nó 14           (Índice 15  ) | Sem alteração (0.000)
✏️ Nó 15           (Índice 16  ) | 0.005 → 0.007
⏭️ Nó 16           (Índice 17  ) | Sem alteração (0.000)
✏️ Nó 17           (Índice 18  ) | 0.022 → 0.031
⏭️ Nó 18           (Índice 19  ) | Sem

In [403]:
# # cria um dicionário apenas com os nós que existem na rede
# dict_novas_demandas_filtrado = {id_nome: demanda 
#                                 for id_nome, demanda in dict_novas_demandas.items() 
#                                 if id_nome in dict_name_index}

# # agora altera apenas esses nós
# for id_nome, new_demand in dict_novas_demandas_filtrado.items():
#     id_index = dict_name_index[id_nome]
#     demanda_atual = d.getNodeBaseDemands(id_index)
#     d.setNodeBaseDemands(id_index, new_demand)
    
#     print(f"Nó {id_nome} (Índice {id_index}) - Demanda ajustada")
#     print(f"Demanda antiga: {demanda_atual}, Nova demanda: {new_demand}")


In [48]:
# Calcular a demanda total da rede com base no formato correto retornado
demanda_total = 0

for i in range(d.getNodeCount()):
    demandas = d.getNodeBaseDemands(i)

    if isinstance(demandas, dict):
        for val in demandas.values():
            if isinstance(val, (list, tuple, np.ndarray)):
                demanda_total += sum(val)
            elif isinstance(val, (int, float)):
                demanda_total += val
    else:
        print(f"Aviso: formato inesperado na demanda do nó {i} → {demandas}")

print("📊 Demanda total da rede:", demanda_total)


📊 Demanda total da rede: 472.19683656678535


c:\Users\filipe_silva\AppData\Local\anaconda3\Lib\site-packages\epyt\epanet.py:14411: UserWarning: Error 203: function call contains undefined node
  warnings.warn(errmssg.value.decode())


In [49]:
d.saveInputFile("Epanet Gerados\\Recanto_Riacho2\\Reborn\\RCE RF2 Reborn2.inp")

## Redução de demanda para alocar em outro lugar (Por porcentagem)

In [ ]:
#Testando pela elevação se foi pego no nó correto
d.getNodeBaseDemands(dict_ids['30'])

{1: array([0.10794278])}

In [ ]:
# Criar o dicionário de IDs (nome -> índice)
dict_ids = {nome: index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}

# Inicializar as variáveis de soma
soma_demanda_antiga = 0
soma_demanda_nova = 0
soma_reduzida = 0

# Porcentagem de redução (ajustável)
percentual_reducao = 0.025

# Percorrer os nós da lista
for id in list_NODENUM:
    if str(id) in dict_ids:
        index = dict_ids[str(id)]

        # Obter as demandas atuais do nó, garantindo que são floats
        demandas_no = d.getNodeBaseDemands(index).values()

        # Converter qualquer array multidimensional para um valor escalar
        demanda_atual = sum(float(valor.item() if hasattr(valor, 'item') else float(valor)) for valor in demandas_no)

        # Calcular a nova demanda (redução em %)
        nova_demanda = demanda_atual * (1 - percentual_reducao)

        # Atualizar a demanda do nó
        d.setNodeBaseDemands(index, nova_demanda)

        # Atualizar as somas
        soma_demanda_antiga += demanda_atual
        soma_demanda_nova += nova_demanda
        soma_reduzida += (demanda_atual - nova_demanda)

        print(f"Nó {id} (Índice {index}) - Demanda ajustada")
        print(f"Demanda antiga: {demanda_atual}, Nova demanda: {nova_demanda}")

# Exibir o resumo final
print("\n🔎 Resumo Final:")
print(f"Soma total das demandas antigas: {soma_demanda_antiga:.2f}")
print(f"Soma total das demandas novas: {soma_demanda_nova:.2f}")
print(f"Redução total aplicada: {soma_reduzida:.2f}")

    



Nó 18 (Índice 892) - Demanda ajustada
Demanda antiga: 0.07905302941799164, Nova demanda: 0.07707670368254184
Nó 19 (Índice 973) - Demanda ajustada
Demanda antiga: 0.06905302405357361, Nova demanda: 0.06732669845223427
Nó 21 (Índice 1139) - Demanda ajustada
Demanda antiga: 0.05231601372361183, Nova demanda: 0.05100811338052153
Nó 24 (Índice 1796) - Demanda ajustada
Demanda antiga: 0.005579003598541021, Nova demanda: 0.005439528508577496
Nó 25 (Índice 1383) - Demanda ajustada
Demanda antiga: 0.03115800768136978, Nova demanda: 0.030379057489335535
Nó 27 (Índice 1795) - Demanda ajustada
Demanda antiga: 0.005579003598541021, Nova demanda: 0.005439528508577496
Nó 28 (Índice 1398) - Demanda ajustada
Demanda antiga: 0.03115800768136978, Nova demanda: 0.030379057489335535
Nó 30 (Índice 638) - Demanda ajustada
Demanda antiga: 0.11579003930091858, Nova demanda: 0.11289528831839561
Nó 33 (Índice 1790) - Demanda ajustada
Demanda antiga: 0.005579003598541021, Nova demanda: 0.005439528508577496
Nó 36

In [ ]:
#Testando pela elevação se foi pego no nó correto
d.getNodeElevations(dict_ids['18'])

In [ ]:
#Exportando novo arquivo Epanet
d.saveInputFile('Epanet Gerados\Gama\Teste 2,5prct.inp')

<>:1: SyntaxWarning: invalid escape sequence '\G'
<>:1: SyntaxWarning: invalid escape sequence '\G'
C:\Users\filipe_silva\AppData\Local\Temp\ipykernel_33768\1943817669.py:1: SyntaxWarning: invalid escape sequence '\G'
  d.saveInputFile('Epanet Gerados\Gama\Teste 2,5prct.inp')


# Redução de demanda com valor fixo de redução

## Redução igualitaria (Todos reduzem igual)

In [82]:
# Não está performando bem

# # Criar o dicionário de IDs (nome -> índice)
# dict_ids = {nome: index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}

# # Inicializar as variáveis de soma
# soma_demanda_antiga = 0
# soma_demanda_nova = 0
# soma_reduzida = 0

# # Valor total de demanda a ser reduzido (ajustável)
# reduzir_total = 0.5

# # Armazenar os nós válidos com suas demandas
# nos_validos = []

# for id in list_NODENUM:
#     if str(id) in dict_ids:
#         index = dict_ids[str(id)]
#         demandas_no = d.getNodeBaseDemands(index).values()
#         demanda_atual = sum(float(valor.item() if hasattr(valor, 'item') else float(valor)) for valor in demandas_no)
        
#         if demanda_atual > 0:
#             nos_validos.append((id, index, demanda_atual))
#             soma_demanda_antiga += demanda_atual

# # Distribuir a redução
# while reduzir_total > 0 and any(demanda > 0 for _, _, demanda in nos_validos):
#     ativos = [(id, idx, dmd) for id, idx, dmd in nos_validos if dmd > 0]
#     n = len(ativos)
#     if n == 0:
#         break
    
#     parcela = reduzir_total / n
#     novos_nos = []

#     for id, idx, dmd in ativos:
#         remover = min(parcela, dmd)
#         nova_dmd = dmd - remover
#         d.setNodeBaseDemands(idx, nova_dmd)

#         print(f"Nó {id} (Índice {idx}) - Demanda ajustada")
#         print(f"Demanda antiga: {dmd}, Nova demanda: {nova_dmd}")

#         soma_demanda_nova += nova_dmd
#         soma_reduzida += remover
#         reduzir_total -= remover

#         novos_nos.append((id, idx, nova_dmd))
    
#     nos_validos = novos_nos

# # Exibir o resumo final
# print("\n🔎 Resumo Final:")
# print(f"Soma total das demandas antigas: {soma_demanda_antiga:.2f}")
# print(f"Soma total das demandas novas: {soma_demanda_nova:.2f}")
# print(f"Redução total aplicada: {soma_reduzida:.2f}")


## Redução proporcional (quem tem mais, perder mais)

In [591]:
# Criar o dicionário de IDs (nome -> índice)
dict_ids = {nome: index for nome, index in zip(d.getNodeNameID(), d.getNodeIndex())}

# Inicializar as variáveis de soma
soma_demanda_antiga = 0
soma_demanda_nova = 0
soma_reduzida = 0

# Valor total de demanda a ser reduzido (ajustável)
reduzir_total = 0.3

# Armazenar os nós válidos com suas demandas
nos_validos = []

for id in list_NODENUM:
    if str(id) in dict_ids:
        index = dict_ids[str(id)]
        demandas_no = d.getNodeBaseDemands(index).values()
        demanda_atual = sum(float(valor.item() if hasattr(valor, 'item') else float(valor)) for valor in demandas_no)
        
        if demanda_atual > 0:
            nos_validos.append((id, index, demanda_atual))
            soma_demanda_antiga += demanda_atual

# Soma total das demandas que podem ser reduzidas
soma_demandas_positivas = sum(dmd for _, _, dmd in nos_validos)

# Aplicar a redução proporcional
for id, idx, dmd in nos_validos:
    proporcao = dmd / soma_demandas_positivas
    remover = proporcao * reduzir_total
    nova_dmd = max(0, dmd - remover)  # Garante que não vai ficar negativo
    
    d.setNodeBaseDemands(idx, nova_dmd)

    print(f"Nó {id} (Índice {idx}) - Demanda ajustada")
    print(f"Demanda antiga: {dmd:.4f}, Nova demanda: {nova_dmd:.4f}, Redução: {remover:.4f}")

    soma_demanda_nova += nova_dmd
    soma_reduzida += (dmd - nova_dmd)

# Exibir o resumo final
print("\n🔎 Resumo Final:")
print(f"Soma total das demandas antigas: {soma_demanda_antiga:.4f}")
print(f"Soma total das demandas novas: {soma_demanda_nova:.4f}")
print(f"Redução total aplicada: {soma_reduzida:.4f}")


Nó 1322 (Índice 1238) - Demanda ajustada
Demanda antiga: 0.0171, Nova demanda: 0.0160, Redução: 0.0011
Nó 1323 (Índice 1239) - Demanda ajustada
Demanda antiga: 0.0267, Nova demanda: 0.0250, Redução: 0.0017
Nó 1324 (Índice 1240) - Demanda ajustada
Demanda antiga: 0.0027, Nova demanda: 0.0025, Redução: 0.0002
Nó 1394 (Índice 1310) - Demanda ajustada
Demanda antiga: 0.0007, Nova demanda: 0.0006, Redução: 0.0000
Nó 1395 (Índice 1311) - Demanda ajustada
Demanda antiga: 0.0157, Nova demanda: 0.0147, Redução: 0.0010
Nó 1465 (Índice 1381) - Demanda ajustada
Demanda antiga: 0.0172, Nova demanda: 0.0161, Redução: 0.0011
Nó 1466 (Índice 1382) - Demanda ajustada
Demanda antiga: 0.0256, Nova demanda: 0.0240, Redução: 0.0016
Nó 1469 (Índice 1385) - Demanda ajustada
Demanda antiga: 0.0033, Nova demanda: 0.0031, Redução: 0.0002
Nó 1470 (Índice 1386) - Demanda ajustada
Demanda antiga: 0.0046, Nova demanda: 0.0043, Redução: 0.0003
Nó 1471 (Índice 1387) - Demanda ajustada
Demanda antiga: 0.0172, Nova dem

In [594]:
# d.saveInputFile('Epanet Gerados\\Samambaia\\SAM V16.inp')
d.saveInputFile('Epanet Gerados\\Gama2\\GAM2 V35.inp')


In [ ]:
# d.saveInputFile("Epanet Gerados\\Paranoa_Itapoa\\Nova calibracao\\PRN_ITP_Versao reset Itapoa (Nova).inp")